## How to read the results

- **Validation, not proof:** clustering diagnostics show whether a segmentation is reasonably separated and stable; they do not make the labels objectively “true.”
- **Affinity is not coupon impact:** pre-campaign category affinity shows what a segment normally buys; it does not show that a coupon caused additional spending.
- **Matched comparison, not random assignment:** segment–category spending evidence compares recipients with similar non-recipients, but campaigns contain multiple coupons and assignment was not random. Treat the resulting combinations as controlled-pilot candidates.
- **Conditional causal estimate:** campaign lift uses matched difference-in-differences. It is more credible than a raw before/after comparison but still relies on comparable trends and observed spending history.
- **Scenario, not accounting truth:** ROI depends heavily on uplift, margin, and mailing-cost assumptions. Treat worst/base/best cases as planning ranges.
- **Panel caveat:** households enter the panel at different times, demographics are missing non-randomly, and purchases outside the retailer are unobserved.

## 0. Setup and data loading

Required packages: `pandas`, `pyarrow`, `numpy`, `scipy`, `scikit-learn`, `statsmodels`, and `matplotlib`. The optional promotion cell also needs `duckdb`.

If the selected notebook kernel is missing them, run this in a temporary cell and then restart the kernel:

```python
%pip install pandas pyarrow numpy scipy scikit-learn statsmodels matplotlib duckdb
```

Package installation is left explicit rather than happening automatically inside the analysis.

In [ ]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
warnings.filterwarnings("ignore", category=FutureWarning)
plt.style.use("seaborn-v0_8-whitegrid")

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / "data" / "processed").is_dir()),
    cwd,
)
PROC = PROJECT_ROOT / "data" / "processed"

raw_files = [
    "transaction_data.csv",
    "product.csv",
    "hh_demographic.csv",
    "campaign_desc.csv",
    "campaign_table.csv",
    "coupon.csv",
    "coupon_redempt.csv",
]
raw_candidates = [
    PROJECT_ROOT / "archive (1)",
    PROJECT_ROOT / "data" / "raw",
    Path.home() / "Downloads" / "archive (1)",
    Path.home() / "Downloads" / "datasets",  # legacy fallback
]
RAW_DATA_DIR = next(
    (
        path
        for path in raw_candidates
        if all((path / filename).is_file() for filename in raw_files)
    ),
    max(
        raw_candidates,
        key=lambda path: sum((path / filename).is_file() for filename in raw_files),
    ),
)

# Enabled for the complete executed notebook. Set to False for quicker reruns.
RUN_PROMO = True

NONSHOP = ["KIOSK-GAS", "MISC SALES TRAN", "MISC. TRANS."]
MARGIN = 0.22       # planning assumption: approximate company-wide gross margin, not an observed category margin
DISC_YR = 0.10      # annual discount rate
HORIZON_Q = 12      # 12 quarters = 3 years

def hdr(title):
    print("\n" + "=" * 8 + " " + title + " " + "=" * 8)

required = [
    *(RAW_DATA_DIR / filename for filename in raw_files),
    PROC / "df_household_campaigns.parquet",
    PROC / "df_coupon_redemptions.parquet",
    PROC / "df_coupon_product_mapping.parquet",
    PROC / "df_store_profiles.parquet",
    PROC / "df_promo_panel.parquet",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))



The transaction table is enriched with product department, brand, and commodity. `shop` excludes fuel and miscellaneous transaction lines so customer value and shopping behavior are not dominated by non-merchandise activity.

In [ ]:
t = pd.read_csv(RAW_DATA_DIR / "transaction_data.csv")
prod = pd.read_csv(RAW_DATA_DIR / "product.csv")
demo = pd.read_csv(RAW_DATA_DIR / "hh_demographic.csv")
hc = pd.read_parquet(PROC / "df_household_campaigns.parquet")
cr = pd.read_parquet(PROC / "df_coupon_redemptions.parquet")
cm = pd.read_parquet(PROC / "df_coupon_product_mapping.parquet")
sp = pd.read_parquet(PROC / "df_store_profiles.parquet")

t = t.merge(
    prod[["PRODUCT_ID", "DEPARTMENT", "BRAND", "COMMODITY_DESC"]],
    on="PRODUCT_ID",
    how="left",
)
shop = t[~t.DEPARTMENT.isin(NONSHOP)].copy()

display(pd.DataFrame({
    "rows": [len(t), len(prod), len(demo), len(hc), len(cr), len(cm), len(sp), len(shop)],
    "columns": [t.shape[1], prod.shape[1], demo.shape[1], hc.shape[1], cr.shape[1], cm.shape[1], sp.shape[1], shop.shape[1]],
}, index=["transactions", "products", "demographics", "household_campaigns", "coupon_redemptions", "coupon_mapping", "store_profiles", "shopping_transactions"]))

## 1. Dataset introduction and business problem

The Complete Journey data records two years of transactions for 2,500 Kroger households, together with product, demographic, campaign, coupon-redemption and product–store–week promotion information. This section explains the structure, coverage and limitations of those sources before any modelling.

### Business problem

Type B/C campaigns distribute the same complete coupon bundle to every household assigned to a campaign. The business problem is whether Kroger can improve this approach by identifying customer segments whose spending genuinely responds to campaigns, rather than relying on coupon redemption alone.

> **How should Kroger distribute Type B/C coupon campaigns more selectively to valuable and campaign-responsive customer segments, while using display and mailer findings as supplementary promotional guidance?**

### 1.1 Dataset structure and field definitions

In [ ]:
# Raw marketing files are loaded here only for the presentation-ready data dictionary.
campaign_desc_intro = pd.read_csv(RAW_DATA_DIR / "campaign_desc.csv")
campaign_assignments_intro = pd.read_csv(RAW_DATA_DIR / "campaign_table.csv")
coupon_mapping_intro = pd.read_csv(RAW_DATA_DIR / "coupon.csv")
coupon_redemptions_intro = pd.read_csv(RAW_DATA_DIR / "coupon_redempt.csv")

import pyarrow.parquet as pq
promotion_panel_path = PROC / "df_promo_panel.parquet"
promotion_panel_rows = pq.ParquetFile(promotion_panel_path).metadata.num_rows

dataset_inventory = pd.DataFrame([
    {
        "Data family": "Transactions",
        "Source": "transaction_data.csv",
        "Rows": len(t),
        "Observation unit": "One product line within a shopping basket",
        "Coverage": f"{t.household_key.nunique():,} households; {t.BASKET_ID.nunique():,} baskets; {t.STORE_ID.nunique():,} stores",
    },
    {
        "Data family": "Products",
        "Source": "product.csv",
        "Rows": len(prod),
        "Observation unit": "One product ID",
        "Coverage": f"{prod.DEPARTMENT.nunique():,} departments; {prod.COMMODITY_DESC.nunique():,} commodities; {prod.SUB_COMMODITY_DESC.nunique():,} sub-commodities",
    },
    {
        "Data family": "Household demographics",
        "Source": "hh_demographic.csv",
        "Rows": len(demo),
        "Observation unit": "One household with available demographics",
        "Coverage": f"{len(demo):,} of {t.household_key.nunique():,} observed households ({len(demo) / t.household_key.nunique():.1%})",
    },
    {
        "Data family": "Campaign definitions",
        "Source": "campaign_desc.csv",
        "Rows": len(campaign_desc_intro),
        "Observation unit": "One campaign",
        "Coverage": f"{campaign_desc_intro.CAMPAIGN.nunique():,} campaigns; {campaign_desc_intro.DESCRIPTION.nunique()} documented delivery types",
    },
    {
        "Data family": "Campaign assignments",
        "Source": "campaign_table.csv",
        "Rows": len(campaign_assignments_intro),
        "Observation unit": "One targeted household–campaign assignment",
        "Coverage": f"{campaign_assignments_intro.household_key.nunique():,} targeted households",
    },
    {
        "Data family": "Coupon mapping",
        "Source": "coupon.csv",
        "Rows": len(coupon_mapping_intro),
        "Observation unit": "One coupon–eligible-product mapping within a campaign",
        "Coverage": f"{coupon_mapping_intro.COUPON_UPC.nunique():,} coupon codes; {coupon_mapping_intro.PRODUCT_ID.nunique():,} eligible products",
    },
    {
        "Data family": "Coupon redemptions",
        "Source": "coupon_redempt.csv",
        "Rows": len(coupon_redemptions_intro),
        "Observation unit": "One recorded coupon redemption",
        "Coverage": f"{coupon_redemptions_intro.household_key.nunique():,} redeeming households",
    },
    {
        "Data family": "Product promotion placements",
        "Source": "df_promo_panel.parquet",
        "Rows": promotion_panel_rows,
        "Observation unit": "One observed product–store–week placement row",
        "Coverage": "Mailer-only, display-only or display + mailer; no neither-placement rows",
    },
])

display(dataset_inventory.style.hide(axis="index").format({"Rows": "{:,.0f}"}).set_properties(
    subset=["Data family", "Source", "Observation unit", "Coverage"],
    **{"text-align": "left", "white-space": "normal", "max-width": "360px"},
).set_caption("Dataset inventory and observation units"))

transaction_dictionary = pd.DataFrame([
    ["Identifiers", "household_key, BASKET_ID, PRODUCT_ID, STORE_ID", "Identify the household, basket, purchased product and store"],
    ["Relative time", "DAY, WEEK_NO, TRANS_TIME", "Relative transaction day and week plus recorded transaction time"],
    ["Quantity", "QUANTITY", "Number of units recorded for the product line"],
    ["Sales value", "SALES_VALUE", "Final dollar value recorded for the product line after applicable discounts"],
    ["Retail discount", "RETAIL_DISC", "Store price markdown; does not necessarily involve a coupon"],
    ["Coupon discount", "COUPON_DISC", "Value deducted through a redeemed coupon"],
    ["Coupon-match discount", "COUPON_MATCH_DISC", "Additional store-funded reduction on top of the coupon; not present for every coupon"],
], columns=["Information", "Fields", "Meaning"])
display(transaction_dictionary.style.hide(axis="index").set_properties(
    subset=["Information", "Fields", "Meaning"],
    **{"text-align": "left", "white-space": "normal", "max-width": "430px"},
).set_caption(
    "Transaction information — one row represents one product line within a shopping basket. "
    "Discount amounts are stored as zero or negative dollar values."
))

department_examples = [
    value for value in ["GROCERY", "PRODUCE", "MEAT", "DELI", "DRUG GM", "NUTRITION", "PASTRY"]
    if value in set(prod.DEPARTMENT.dropna())
]
commodity_examples = prod.loc[
    ~prod.COMMODITY_DESC.isin(["NO COMMODITY DESCRIPTION"]), "COMMODITY_DESC"
].value_counts().head(5).index.tolist()
subcommodity_examples = prod.loc[
    ~prod.SUB_COMMODITY_DESC.isin(["NO SUBCOMMODITY DESCRIPTION"]), "SUB_COMMODITY_DESC"
].value_counts().head(5).index.tolist()
size_examples = (
    prod.CURR_SIZE_OF_PRODUCT.fillna("").astype(str).str.strip()
    .loc[lambda values: values.ne("")].drop_duplicates().head(6).tolist()
)
product_dictionary = pd.DataFrame([
    ["Department", "Broad retail division", ", ".join(department_examples)],
    ["Commodity", "General product category", ", ".join(commodity_examples)],
    ["Sub-commodity", "More detailed product type", ", ".join(subcommodity_examples)],
    ["Brand", "National or private-label classification", ", ".join(prod.BRAND.dropna().astype(str).unique())],
    ["Current package size", "Recorded package size or count", ", ".join(size_examples)],
], columns=["Product information", "Meaning", "Observed examples"])
display(product_dictionary.style.hide(axis="index").set_properties(
    subset=["Product information", "Meaning", "Observed examples"],
    **{"text-align": "left", "white-space": "normal", "max-width": "440px"},
).set_caption("Product information available for each product ID"))

demographic_dictionary = pd.DataFrame([
    ["Age group", "AGE_DESC"],
    ["Income range", "INCOME_DESC"],
    ["Marital status", "MARITAL_STATUS_CODE"],
    ["Homeownership status", "HOMEOWNER_DESC"],
    ["Household composition", "HH_COMP_DESC"],
    ["Household size", "HOUSEHOLD_SIZE_DESC"],
    ["Presence/number of children", "KID_CATEGORY_DESC"],
], columns=["Household demographic information", "Field"])
display(demographic_dictionary.style.hide(axis="index").set_properties(
    subset=["Household demographic information", "Field"], **{"text-align": "left"},
).set_caption(
    f"Household demographics — available for {len(demo):,} of {t.household_key.nunique():,} households "
    f"({len(demo) / t.household_key.nunique():.1%})"
))

coupon_product_counts_intro = coupon_mapping_intro.groupby(["CAMPAIGN", "COUPON_UPC"]).PRODUCT_ID.nunique()
marketing_dictionary = pd.DataFrame([
    ["Campaign assignments", "Which households participated in each campaign, together with campaign type and duration"],
    ["Type A delivery", "Each household received 16 coupons selected from the campaign pool using prior purchase behaviour; the household-specific 16 are unavailable"],
    ["Type B/C delivery", "Every household assigned to the campaign received all coupons listed for that campaign"],
    ["Coupon mapping", "Which products were eligible for each coupon; for Type A this is the possible pool, while for Types B/C it is the issued campaign bundle"],
    ["Coupon redemptions", "Which household redeemed which coupon under which campaign and on what relative day"],
], columns=["Household-level marketing information", "Meaning"])
display(marketing_dictionary.style.hide(axis="index").set_properties(
    subset=["Household-level marketing information", "Meaning"],
    **{"text-align": "left", "white-space": "normal", "max-width": "650px"},
).set_caption(
    f"Campaign and coupon information — {(coupon_product_counts_intro > 1).mean():.1%} of coupon–campaign pairs "
    "map to more than one eligible product"
))

campaign_type_intro = (
    campaign_desc_intro.assign(
        duration_days=lambda frame: frame.END_DAY - frame.START_DAY + 1
    )
    .merge(campaign_assignments_intro.groupby("CAMPAIGN").household_key.nunique().rename("targeted_households"), on="CAMPAIGN")
    .merge(coupon_mapping_intro.groupby("CAMPAIGN").COUPON_UPC.nunique().rename("coupon_codes"), on="CAMPAIGN")
    .groupby("DESCRIPTION").agg(
        campaigns=("CAMPAIGN", "nunique"),
        median_targeted_households=("targeted_households", "median"),
        median_coupon_codes=("coupon_codes", "median"),
        median_duration_days=("duration_days", "median"),
        maximum_duration_days=("duration_days", "max"),
    ).reindex(["TypeA", "TypeB", "TypeC"])
)
display(campaign_type_intro.style.format({
    "campaigns": "{:,.0f}", "median_targeted_households": "{:,.0f}",
    "median_coupon_codes": "{:,.0f}", "median_duration_days": "{:.0f}",
    "maximum_duration_days": "{:.0f}",
}).set_caption(
    "Observed campaign profile. Type A coupon counts are pool sizes; Type B/C counts are complete issued bundles."
))

promotion_dictionary = pd.DataFrame([
    ["Placement key", "PRODUCT_ID, STORE_ID, WEEK_NO", "Product, store and week of observed promotional placement"],
    ["Display placement", "has_display", "1 when display support was recorded; otherwise 0"],
    ["Mailer placement", "has_mailer", "1 when mailer support was recorded; otherwise 0"],
    ["Combined support", "has_both_display_mailer", "1 when display and mailer support occurred together"],
], columns=["Store-level product promotion information", "Fields", "Meaning"])
display(promotion_dictionary.style.hide(axis="index").set_properties(
    subset=["Store-level product promotion information", "Fields", "Meaning"],
    **{"text-align": "left", "white-space": "normal", "max-width": "430px"},
).set_caption(
    "Store-level product promotion information — placement is product–store–week context and is independent "
    "of whether a coupon discount was redeemed"
))

### 1.2 Coverage and analysis context

In [ ]:
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    hdr("0. DATA FACTS")
    hh = t.groupby("household_key").agg(first=("DAY", "min"), last=("DAY", "max"), spend=("SALES_VALUE", "sum"))
    hh["has_demo"] = hh.index.isin(demo.household_key)
    print(f"households {len(hh)}, with demographics {hh.has_demo.sum()} ({hh.has_demo.mean():.0%})")
    print("median 2-yr spend with / without demographics: $%.0f / $%.0f" % (
        hh.spend[hh.has_demo].median(), hh.spend[~hh.has_demo].median()))
    print("first purchase day: median %d, 95th pct %d (panel ramp-up)" % (hh["first"].median(), hh["first"].quantile(.95)))
    print("bought in last 30 days: %.0f%%" % (100 * ((t.DAY.max() - hh["last"]) <= 30).mean()))
    dep = t.groupby("DEPARTMENT")[["SALES_VALUE", "QUANTITY"]].sum()
    print("KIOSK-GAS share of revenue %.1f%%, of units %.1f%%" % (
        100 * dep.SALES_VALUE["KIOSK-GAS"] / dep.SALES_VALUE.sum(), 100 * dep.QUANTITY["KIOSK-GAS"] / dep.QUANTITY.sum()))
    red = cr.groupby(["household_key", "CAMPAIGN"]).size().rename("n_red").reset_index()
    pairs = hc.merge(red, on=["household_key", "CAMPAIGN"], how="left"); pairs["y"] = pairs.n_red.notna().astype(int)
    print(f"campaigns {hc.CAMPAIGN.nunique()}, households reached {hc.household_key.nunique()}, pairs {len(hc)}, "
          f"first campaign starts day {hc.START_DAY.min()}")
    print("redemption rate by type:", (pairs.groupby("campaign_type").y.mean() * 100).round(1).to_dict(),
          "overall %.1f%%" % (100 * pairs.y.mean()))
    ok = sp[~sp.sparse_households_flag_lt_20]
    print(f"stores with >=20 panel households: {len(ok)} of {len(sp)}, "
          f"{100 * ok.total_observed_sales_revenue.sum() / sp.total_observed_sales_revenue.sum():.1f}% of panel revenue")

In [ ]:
data_fact_summary = pd.DataFrame({
    "Measure": [
        "Observed households",
        "Households with demographics",
        "Median two-year spend: demographics available",
        "Median two-year spend: demographics unavailable",
        "Median first observed purchase day",
        "Households purchasing in final 30 days",
        "KIOSK-GAS share of revenue",
        "KIOSK-GAS share of recorded units",
        "Campaigns / reached households / household–campaign pairs",
        "Stores with at least 20 panel households",
        "Panel revenue represented by those stores",
    ],
    "Result": [
        f"{len(hh):,}",
        f"{hh.has_demo.sum():,} ({hh.has_demo.mean():.1%})",
        f"${hh.spend[hh.has_demo].median():,.0f}",
        f"${hh.spend[~hh.has_demo].median():,.0f}",
        f"Day {hh['first'].median():.0f}",
        f"{((t.DAY.max() - hh['last']) <= 30).mean():.1%}",
        f"{dep.SALES_VALUE['KIOSK-GAS'] / dep.SALES_VALUE.sum():.1%}",
        f"{dep.QUANTITY['KIOSK-GAS'] / dep.QUANTITY.sum():.1%}",
        f"{hc.CAMPAIGN.nunique()} / {hc.household_key.nunique():,} / {len(hc):,}",
        f"{len(ok)} of {len(sp)}",
        f"{ok.total_observed_sales_revenue.sum() / sp.total_observed_sales_revenue.sum():.1%}",
    ],
})
campaign_redemption_summary = pairs.groupby("campaign_type")["y"].agg(
    household_campaign_pairs="size",
    redeemed_pairs="sum",
    redemption_rate="mean",
)
display(data_fact_summary.style.hide(axis="index").set_caption("Dataset exploration and coverage summary"))
display(campaign_redemption_summary.style.format({
    "household_campaign_pairs": "{:,.0f}",
    "redeemed_pairs": "{:,.0f}",
    "redemption_rate": "{:.1%}",
}).set_caption("Observed household–campaign redemption by documented campaign type"))

## 2. Diagnose past campaigns and coupons

This section uses the raw campaign files and official delivery definitions to establish how the campaign types differ. It distinguishes two questions:

1. **Portfolio and reach:** how many households and days does each campaign cover, and how much do recipient audiences overlap?
2. **Observed use:** which listed coupon codes were redeemed, how concentrated was redemption, and when did it occur?

The diagnostics also establish two facts that motivate the later analysis: a coupon can apply to multiple eligible products and categories, and observed redemption rates vary substantially across campaigns and campaign types.

For Type A, coupon and product counts describe the available personalisation pool, not the 16 coupons received by a specific household. For Types B/C, they describe the complete issued bundle. Exact duplicate coupon–product mappings are removed, and every redemption must match a campaign participant, a coupon listed for that campaign, and the campaign window.

In [ ]:
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    import statsmodels.formula.api as smf
    from statsmodels.discrete.conditional_models import ConditionalLogit
    from scipy.stats import wilcoxon, mannwhitneyu

    # Keep raw campaign tables separate from the prepared parquet objects used later.
    desc = pd.read_csv(RAW_DATA_DIR / "campaign_desc.csv").rename(columns={"DESCRIPTION": "type"})
    table = pd.read_csv(RAW_DATA_DIR / "campaign_table.csv").rename(columns={"DESCRIPTION": "type"})
    coupon = pd.read_csv(RAW_DATA_DIR / "coupon.csv")
    red = pd.read_csv(RAW_DATA_DIR / "coupon_redempt.csv")
    coupon_raw_rows = len(coupon)

    hdr("2.1 DATA CHECKS")
    print(f"campaigns {len(desc)}, household-campaign pairs {len(table):,}, households targeted {table.household_key.nunique():,}")
    print(f"coupon.csv rows {len(coupon):,}; exact duplicate rows {coupon.duplicated().sum():,} -> dropped")
    coupon = coupon.drop_duplicates()
    print(f"  unique coupons {coupon.COUPON_UPC.nunique():,}, products {coupon.PRODUCT_ID.nunique():,}")
    print(f"redemptions {len(red):,}; exact duplicates {red.duplicated().sum()}")
    red = red.drop_duplicates().merge(desc, on="CAMPAIGN", how="left")
    chk_member = red.merge(table[["household_key", "CAMPAIGN"]], how="left", indicator=True, on=["household_key", "CAMPAIGN"])
    chk_coupon = red.merge(coupon[["COUPON_UPC", "CAMPAIGN"]].drop_duplicates(), how="left", indicator=True, on=["COUPON_UPC", "CAMPAIGN"])
    print(f"  redemptions by households not in that campaign: {(chk_member._merge == 'left_only').sum()}")
    print(f"  redemptions of coupons not listed for that campaign: {(chk_coupon._merge == 'left_only').sum()}")
    print(f"  redemptions outside the campaign window: {((red.DAY < red.START_DAY) | (red.DAY > red.END_DAY)).sum()}")
    assert (table.merge(desc, on="CAMPAIGN").pipe(lambda d: (d.type_x == d.type_y).all())), "campaign type mismatch"

In [ ]:
campaign_data_checks = pd.DataFrame({
    "Check": [
        "Campaign definitions",
        "Household–campaign assignments",
        "Targeted households",
        "Duplicate coupon–product mappings removed",
        "Unique coupon codes after cleaning",
        "Eligible products",
        "Redemption records",
        "Redemptions by non-assigned households",
        "Coupons not listed for the campaign",
        "Redemptions outside campaign window",
    ],
    "Result": [
        f"{len(desc):,}",
        f"{len(table):,}",
        f"{table.household_key.nunique():,}",
        f"{coupon_raw_rows - len(coupon):,}",
        f"{coupon.COUPON_UPC.nunique():,}",
        f"{coupon.PRODUCT_ID.nunique():,}",
        f"{len(red):,}",
        f"{(chk_member._merge == 'left_only').sum():,}",
        f"{(chk_coupon._merge == 'left_only').sum():,}",
        f"{((red.DAY < red.START_DAY) | (red.DAY > red.END_DAY)).sum():,}",
    ],
})
display(campaign_data_checks.style.hide(axis="index").set_caption("Campaign-data integrity checks"))

### 2.1 Campaign portfolio and recipient targeting

The dataset contains 30 campaigns grouped into three documented delivery types. According to the official Complete Journey guide, each Type A household received 16 coupons selected from a larger campaign pool using prior purchase behaviour; the household-specific 16 are not recorded. Every Type B and Type C recipient received all coupons listed for that campaign. The analysis below therefore treats Type A coupon counts as **pool size**, while Type B/C counts describe the issued campaign bundle.

This subsection profiles campaign scale, duration and household overlap. Product placement remains product–store–week context rather than one campaign-level treatment. Source: [The Complete Journey User Guide — coupons](https://bradleyboehmke.github.io/completejourney/articles/completejourney.html#coupons).

In [ ]:
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    hdr("2.2 CAMPAIGN LEVEL - is TypeA better, or just bigger?")
    cam = desc.set_index("CAMPAIGN")
    cam["days"] = cam.END_DAY - cam.START_DAY + 1
    cam["recipients"] = table.groupby("CAMPAIGN").household_key.nunique()
    cam = cam.join(coupon.groupby("CAMPAIGN").agg(coupons=("COUPON_UPC", "nunique"), products=("PRODUCT_ID", "nunique")))
    cam = cam.join(red.groupby("CAMPAIGN").agg(redeemers=("household_key", "nunique"), redemptions=("COUPON_UPC", "size"),
                                               coupons_redeemed=("COUPON_UPC", "nunique"))).fillna(0)
    cam["hh_redeem_rate"] = cam.redeemers / cam.recipients
    cam["redemptions_per_100_coupons_offered"] = 100 * cam.redemptions / (cam.recipients * cam.coupons)
    cam["share_coupons_ever_redeemed"] = cam.coupons_redeemed / cam.coupons
    g = cam.groupby("type")
    agg = pd.DataFrame({
        "campaigns": g.size(), "median_coupons": g.coupons.median(), "median_products": g.products.median(),
        "median_days": g.days.median(), "recipients": g.recipients.sum(), "redeemers": g.redeemers.sum(),
        "redemptions": g.redemptions.sum()})
    agg["hh_redeem_rate"] = agg.redeemers / agg.recipients
    agg["redemptions_per_recipient"] = agg.redemptions / agg.recipients
    agg["redemptions_per_100_coupons_offered"] = 100 * agg.redemptions / (cam.recipients * cam.coupons).groupby(cam.type).sum()
    agg["share_coupons_ever_redeemed"] = g.coupons_redeemed.sum() / g.coupons.sum()
    print(agg.round(3).T.to_string())
    print(f"\ncorrelation across 30 campaigns: log(coupons) vs household redemption rate "
          f"{np.corrcoef(np.log(cam.coupons), cam.hh_redeem_rate)[0, 1]:+.2f}; "
          f"log(days) vs rate {np.corrcoef(np.log(cam.days), cam.hh_redeem_rate)[0, 1]:+.2f}")


In [ ]:
campaign_type_profile = cam.groupby("type").agg(
    campaigns=("recipients", "size"),
    median_households_per_campaign=("recipients", "median"),
    median_coupons_per_campaign=("coupons", "median"),
    median_eligible_products=("products", "median"),
    median_duration_days=("days", "median"),
    total_household_campaign_assignments=("recipients", "sum"),
    total_redeemers=("redeemers", "sum"),
)
campaign_type_profile["household_redemption_rate"] = (
    campaign_type_profile.total_redeemers
    / campaign_type_profile.total_household_campaign_assignments
)
campaign_type_profile = campaign_type_profile.reindex(["TypeA", "TypeB", "TypeC"])
campaign_type_profile["documented_delivery"] = [
    "Each household received 16 coupons selected from the listed pool using prior purchase behaviour; the selected 16 are unavailable",
    "Every assigned household received every coupon listed for the campaign",
    "Every assigned household received every coupon listed for the campaign",
]

display(campaign_type_profile.style.format({
    "campaigns": "{:,.0f}",
    "median_households_per_campaign": "{:,.0f}",
    "median_coupons_per_campaign": "{:,.0f}",
    "median_eligible_products": "{:,.0f}",
    "median_duration_days": "{:.0f}",
    "total_household_campaign_assignments": "{:,.0f}",
    "total_redeemers": "{:,.0f}",
    "household_redemption_rate": "{:.1%}",
}).set_properties(subset=["documented_delivery"], **{
    "text-align": "left", "white-space": "normal", "max-width": "340px"
}).set_caption("Campaign-type profile and documented coupon-delivery mechanics"))

type_colours = ["#2563EB", "#F59E0B", "#10B981"]
chart_specs = [
    ("campaigns", "Number of campaigns", "campaigns"),
    ("median_households_per_campaign", "Median households per campaign", "households"),
    ("median_coupons_per_campaign", "Median listed coupon codes", "codes"),
    ("median_duration_days", "Median campaign duration", "days"),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=130)
for ax, (column, title, unit) in zip(axes.flat, chart_specs):
    values = campaign_type_profile[column]
    bars = ax.bar(campaign_type_profile.index, values, color=type_colours, width=0.62)
    ax.set_title(title, fontsize=12, weight="bold")
    ax.set_ylabel(unit.capitalize())
    ax.bar_label(bars, labels=[f"{v:,.0f}" for v in values], padding=3, fontsize=10)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="x", visible=False)
fig.suptitle("Campaign types differ in delivery mechanism, scale and duration",
             fontsize=16, weight="bold", y=1.01)
fig.text(0.5, 0.005,
         "For Type A, listed coupon codes form a pool and each household received 16; for Types B/C, all listed codes were issued.",
         ha="center", fontsize=9, color="#4B5563")
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()

**Portfolio of all 30 campaigns.** Each coupon can map to one or multiple eligible products. Type A rows describe the available coupon pool—not the unknown household-specific set of 16—whereas Type B/C rows describe the complete bundle received by every campaign recipient. Product placement is recorded separately at the **product–store–week** level, so a single campaign can be associated with mailer-only, display-only and combined display-and-mailer observations across its products, stores and active weeks.

The detailed table and charts below connect these levels without treating placement as a campaign-wide attribute. Placement shares are conditional on rows present in the promotion panel, which contains no neither-display-nor-mailer observations.

In [ ]:
# Campaign portfolio: household reach, coupon/product breadth and placement context.
campaign_coupon_products = coupon.merge(
    prod[["PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC"]],
    on="PRODUCT_ID", how="left", validate="many_to_one",
)
campaign_taxonomy = campaign_coupon_products.groupby("CAMPAIGN").agg(
    departments=("DEPARTMENT", "nunique"),
    commodities=("COMMODITY_DESC", "nunique"),
)
coupon_scope_portfolio = campaign_coupon_products.groupby(["CAMPAIGN", "COUPON_UPC"]).agg(
    eligible_products=("PRODUCT_ID", "nunique"),
    eligible_departments=("DEPARTMENT", "nunique"),
).reset_index()
campaign_scope = coupon_scope_portfolio.groupby("CAMPAIGN").agg(
    multi_product_coupon_share=("eligible_products", lambda values: values.gt(1).mean()),
    multi_department_coupon_share=("eligible_departments", lambda values: values.gt(1).mean()),
)

campaign_placement_counts = pd.DataFrame(columns=[
    "CAMPAIGN", "mailer_only_observations", "display_only_observations", "both_observations"
])
if RUN_PROMO:
    import duckdb
    portfolio_con = duckdb.connect()
    coupon_path = str(RAW_DATA_DIR / "coupon.csv").replace("'", "''")
    campaign_path = str(RAW_DATA_DIR / "campaign_desc.csv").replace("'", "''")
    promo_path = str(PROC / "df_promo_panel.parquet").replace("'", "''")
    campaign_placement_counts = portfolio_con.execute(f'''
        with active_campaign_products as (
            select distinct c.CAMPAIGN, c.PRODUCT_ID, w::integer as WEEK_NO
            from read_csv_auto('{coupon_path}') c
            join read_csv_auto('{campaign_path}') d using (CAMPAIGN)
            cross join range(
                cast(floor((d.START_DAY - 1) / 7) + 1 as bigint),
                cast(floor((d.END_DAY - 1) / 7) + 2 as bigint)
            ) weeks(w)
        )
        select
            a.CAMPAIGN,
            count(*) filter (where not p.has_display and p.has_mailer) as mailer_only_observations,
            count(*) filter (where p.has_display and not p.has_mailer) as display_only_observations,
            count(*) filter (where p.has_display and p.has_mailer) as both_observations
        from active_campaign_products a
        join read_parquet('{promo_path}') p
          on p.PRODUCT_ID = a.PRODUCT_ID and p.WEEK_NO = a.WEEK_NO
        group by a.CAMPAIGN
    ''').df()
    portfolio_con.close()

campaign_portfolio = (
    cam.reset_index()
    .merge(campaign_taxonomy.reset_index(), on="CAMPAIGN", how="left", validate="one_to_one")
    .merge(campaign_scope.reset_index(), on="CAMPAIGN", how="left", validate="one_to_one")
    .merge(campaign_placement_counts, on="CAMPAIGN", how="left", validate="one_to_one")
)
placement_count_columns = [
    "mailer_only_observations", "display_only_observations", "both_observations"
]
campaign_portfolio[placement_count_columns] = campaign_portfolio[placement_count_columns].fillna(0)
campaign_portfolio["placement_observations"] = campaign_portfolio[placement_count_columns].sum(axis=1)
for source, output in [
    ("mailer_only_observations", "mailer_only_share"),
    ("display_only_observations", "display_only_share"),
    ("both_observations", "display_and_mailer_share"),
]:
    campaign_portfolio[output] = (
        campaign_portfolio[source]
        / campaign_portfolio.placement_observations.replace(0, np.nan)
    )

campaign_portfolio_display = campaign_portfolio.rename(columns={
    "CAMPAIGN": "Campaign",
    "type": "Type",
    "START_DAY": "Start day",
    "days": "Duration (days)",
    "recipients": "Targeted households",
    "coupons": "Listed coupon codes*",
    "products": "Eligible products",
    "departments": "Departments",
    "commodities": "Commodities",
    "multi_product_coupon_share": "Multi-product coupons",
    "hh_redeem_rate": "Household redemption rate",
    "mailer_only_share": "Mailer only*",
    "display_only_share": "Display only*",
    "display_and_mailer_share": "Display + mailer*",
})[[
    "Campaign", "Type", "Start day", "Duration (days)", "Targeted households",
    "Listed coupon codes*", "Eligible products", "Departments", "Commodities",
    "Multi-product coupons", "Household redemption rate",
    "Mailer only*", "Display only*", "Display + mailer*",
]].sort_values(["Type", "Start day", "Campaign"])

display(campaign_portfolio_display.style.hide(axis="index").format({
    "Campaign": "{:,.0f}", "Start day": "{:,.0f}", "Duration (days)": "{:,.0f}",
    "Targeted households": "{:,.0f}", "Listed coupon codes*": "{:,.0f}",
    "Eligible products": "{:,.0f}", "Departments": "{:,.0f}", "Commodities": "{:,.0f}",
    "Multi-product coupons": "{:.1%}", "Household redemption rate": "{:.1%}",
    "Mailer only*": "{:.1%}", "Display only*": "{:.1%}", "Display + mailer*": "{:.1%}",
}).set_caption(
    "All 30 campaigns. *For Type A this is pool size; each household received 16 unknown selections. "
    "Placement shares describe observed product–store–week rows for coupon-eligible products during the "
    "campaign window; they are not one campaign-level treatment."
))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), dpi=130)
portfolio_colours = {"TypeA": "#2563EB", "TypeB": "#F59E0B", "TypeC": "#10B981"}
for campaign_type, frame in campaign_portfolio.groupby("type"):
    axes[0].scatter(
        frame.coupons, frame.recipients,
        s=35 + 2.5 * frame.days,
        alpha=0.75, color=portfolio_colours[campaign_type],
        label=campaign_type, edgecolor="white", linewidth=0.6,
    )
    for _, row in frame.iterrows():
        axes[0].annotate(str(int(row.CAMPAIGN)), (row.coupons, row.recipients),
                         xytext=(3, 3), textcoords="offset points", fontsize=7)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("Listed coupon codes (Type A = pool; log scale)")
axes[0].set_ylabel("Targeted households (log scale)")
axes[0].set_title("A. Campaign reach rises with coupon breadth", weight="bold")
axes[0].legend(title="Anonymised type", frameon=False)

placement_by_type = campaign_portfolio.groupby("type")[placement_count_columns].sum()
placement_by_type = placement_by_type.div(placement_by_type.sum(axis=1), axis=0).reindex(
    ["TypeA", "TypeB", "TypeC"]
)
placement_labels = ["Mailer only", "Display only", "Display + mailer"]
placement_colours = ["#93C5FD", "#FBBF24", "#8B5CF6"]
left = np.zeros(len(placement_by_type))
for column, label, colour in zip(placement_count_columns, placement_labels, placement_colours):
    values = placement_by_type[column].fillna(0).values
    bars = axes[1].barh(placement_by_type.index, values, left=left, label=label, color=colour)
    for bar, value, start in zip(bars, values, left):
        if value >= 0.06:
            axes[1].text(start + value / 2, bar.get_y() + bar.get_height() / 2,
                         f"{value:.0%}", ha="center", va="center", fontsize=9)
    left += values
axes[1].set_xlim(0, 1)
axes[1].xaxis.set_major_formatter(PercentFormatter(1))
axes[1].set_xlabel("Share of observed placement rows")
axes[1].set_title("B. Placement mix is product–store–week context", weight="bold")
axes[1].legend(frameon=False, loc="lower center", bbox_to_anchor=(0.5, -0.27), ncol=3)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Thirty campaigns vary in reach, coupon breadth and promotional context",
             fontsize=15, weight="bold", y=1.02)
fig.text(
    0.5, -0.03,
    "Point labels are campaign IDs; point size represents duration. Placement panel has no neither-placement rows.",
    ha="center", fontsize=9, color="#4B5563",
)
plt.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.show()

**Recipient overlap and repeated targeting.** Campaign assignments show who was selected, but not the business rules used for selection. The following exposure profiles describe whether households received only one campaign, several campaigns of the same type, or campaigns from multiple types. This is a targeting-overlap analysis, not the behavioural customer segmentation used in Section 3.

In [ ]:
household_type_counts = table.groupby(["household_key", "type"]).CAMPAIGN.nunique().unstack(fill_value=0)
household_type_counts = household_type_counts.reindex(columns=["TypeA", "TypeB", "TypeC"], fill_value=0)
household_type_counts["total_campaigns"] = household_type_counts.sum(axis=1)
household_type_counts["campaign_types_received"] = (
    household_type_counts[["TypeA", "TypeB", "TypeC"]] > 0
).sum(axis=1)
household_type_counts["repeated_same_type"] = (
    household_type_counts[["TypeA", "TypeB", "TypeC"]] > 1
).any(axis=1)

def exposure_combination(row):
    received = [label[-1] for label in ["TypeA", "TypeB", "TypeC"] if row[label] > 0]
    return "+".join(received) + (" only" if len(received) == 1 else "")

household_type_counts["type_combination"] = household_type_counts.apply(exposure_combination, axis=1)
targeting_overlap_summary = pd.DataFrame({
    "Measure": [
        "Households targeted at least once",
        "Received exactly one campaign",
        "Received multiple campaigns",
        "Received the same campaign type more than once",
        "Received campaigns from multiple types",
        "Received all three campaign types",
        "Median campaigns per targeted household",
        "Maximum campaigns received by one household",
    ],
    "Result": [
        f"{len(household_type_counts):,}",
        f"{(household_type_counts.total_campaigns == 1).sum():,} ({(household_type_counts.total_campaigns == 1).mean():.1%})",
        f"{(household_type_counts.total_campaigns > 1).sum():,} ({(household_type_counts.total_campaigns > 1).mean():.1%})",
        f"{household_type_counts.repeated_same_type.sum():,} ({household_type_counts.repeated_same_type.mean():.1%})",
        f"{(household_type_counts.campaign_types_received > 1).sum():,} ({(household_type_counts.campaign_types_received > 1).mean():.1%})",
        f"{(household_type_counts.campaign_types_received == 3).sum():,} ({(household_type_counts.campaign_types_received == 3).mean():.1%})",
        f"{household_type_counts.total_campaigns.median():.0f}",
        f"{household_type_counts.total_campaigns.max():.0f}",
    ],
})

type_recipient_rows = []
for campaign_type in ["TypeA", "TypeB", "TypeC"]:
    counts = household_type_counts.loc[household_type_counts[campaign_type] > 0, campaign_type]
    type_recipient_rows.append({
        "Campaign type": campaign_type,
        "Campaigns in dataset": int(desc.loc[desc.type == campaign_type, "CAMPAIGN"].nunique()),
        "Unique targeted households": len(counts),
        "Targeted once by this type": int((counts == 1).sum()),
        "Targeted repeatedly by this type": int((counts > 1).sum()),
        "Repeated within-type share": (counts > 1).mean(),
        "Median campaigns of this type": counts.median(),
        "Maximum campaigns of this type": counts.max(),
    })
type_recipient_summary = pd.DataFrame(type_recipient_rows)

combination_order = ["A only", "B only", "C only", "A+B", "A+C", "B+C", "A+B+C"]
exposure_combination_summary = household_type_counts.groupby("type_combination").agg(
    targeted_households=("total_campaigns", "size"),
    average_campaigns_received=("total_campaigns", "mean"),
    median_campaigns_received=("total_campaigns", "median"),
).reindex(combination_order).dropna(how="all")
exposure_combination_summary["share_of_targeted_households"] = (
    exposure_combination_summary.targeted_households / len(household_type_counts)
)

exposure_combination_display = exposure_combination_summary.reset_index().rename(columns={
    "type_combination": "Campaign-type combination",
    "targeted_households": "Targeted households",
    "share_of_targeted_households": "Share of targeted households",
    "average_campaigns_received": "Average campaigns received",
    "median_campaigns_received": "Median campaigns received",
})

display(targeting_overlap_summary.style.hide(axis="index").set_caption(
    "How often targeted households appeared across the 30 campaigns"
))
display(exposure_combination_display.style.hide(axis="index").format({
    "Targeted households": "{:,.0f}",
    "Share of targeted households": "{:.1%}",
    "Average campaigns received": "{:.1f}",
    "Median campaigns received": "{:.0f}",
}).set_caption("Households targeted by each combination of campaign types"))
display(type_recipient_summary.style.hide(axis="index").format({
    "Campaigns in dataset": "{:,.0f}", "Unique targeted households": "{:,.0f}",
    "Targeted once by this type": "{:,.0f}", "Targeted repeatedly by this type": "{:,.0f}",
    "Repeated within-type share": "{:.1%}", "Median campaigns of this type": "{:.1f}",
    "Maximum campaigns of this type": "{:,.0f}",
}).set_caption("Repeated targeting within each anonymised campaign type"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=130)
once = type_recipient_summary.set_index("Campaign type")["Targeted once by this type"]
repeated = type_recipient_summary.set_index("Campaign type")["Targeted repeatedly by this type"]
axes[0].bar(once.index, once, label="Targeted once", color="#93C5FD")
axes[0].bar(repeated.index, repeated, bottom=once, label="Targeted 2+ times", color="#2563EB")
axes[0].set_title("A. Repeat targeting differs by campaign type", weight="bold")
axes[0].set_ylabel("Unique targeted households")
axes[0].legend(frameon=False)

combo_counts = exposure_combination_summary["targeted_households"].astype(int).to_dict()
venn_ax = axes[1]
venn_ax.clear()
circle_specs = [
    ((-0.55, 0.20), "#2563EB", "A"),
    ((0.55, 0.20), "#F59E0B", "B"),
    ((0.00, -0.62), "#10B981", "C"),
]
for centre, colour, label in circle_specs:
    venn_ax.add_patch(plt.Circle(centre, 1.12, facecolor=colour, edgecolor=colour,
                                 linewidth=2, alpha=0.27))

venn_labels = {
    "A only": (-1.12, 0.38),
    "B only": (1.12, 0.38),
    "C only": (0.00, -1.28),
    "A+B": (0.00, 0.78),
    "A+C": (-0.55, -0.55),
    "B+C": (0.55, -0.55),
    "A+B+C": (0.00, -0.05),
}
for combination, (x_pos, y_pos) in venn_labels.items():
    venn_ax.text(
        x_pos, y_pos,
        f"{combination}\n{combo_counts.get(combination, 0):,}",
        ha="center", va="center", fontsize=10,
        weight="bold" if combination == "A+B+C" else "normal",
    )
venn_ax.text(-1.25, 1.23, "Type A", color="#1D4ED8", fontsize=12, weight="bold")
venn_ax.text(0.82, 1.23, "Type B", color="#B45309", fontsize=12, weight="bold")
venn_ax.text(0.00, -1.82, "Type C", color="#047857", fontsize=12, weight="bold", ha="center")
venn_ax.set_xlim(-1.85, 1.85)
venn_ax.set_ylim(-2.00, 1.60)
venn_ax.set_aspect("equal")
venn_ax.axis("off")
venn_ax.set_title("B. Campaign audiences overlapped substantially", weight="bold")
venn_ax.text(
    0, -1.98,
    "Circle areas are illustrative; labels show mutually exclusive household counts.",
    ha="center", va="bottom", fontsize=8.5, color="#4B5563",
)

axes[0].spines[["top", "right"]].set_visible(False)
axes[0].grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

### 2.2 Coupon utilisation, concentration and timing

Coupon-level diagnostics identify codes with no observed redemption, how strongly redemption is concentrated and when first redemption occurs within each campaign window. For Types B and C, all listed coupons were issued to every campaign recipient. For Type A, the listed codes are only the possible pool, so a code with no observed redemption may never have been selected among any household's 16 coupons. “Redeemed” does not measure profitability or causal lift.

In [ ]:
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    hdr("2.6 COUPON LEVEL - wasted coupons, breadth and reuse")
    cp = coupon.groupby("COUPON_UPC").agg(products=("PRODUCT_ID", "nunique"), campaigns=("CAMPAIGN", "nunique"))
    cp["redemptions"] = red.groupby("COUPON_UPC").size()
    cp["redemptions"] = cp.redemptions.fillna(0)
    cp["redeemed"] = cp.redemptions > 0
    print(f"coupons never redeemed: {(~cp.redeemed).sum()} of {len(cp)} ({(~cp.redeemed).mean():.0%})")
    cp["breadth"] = pd.cut(cp.products, [0, 5, 20, 100, np.inf], labels=["1-5 products", "6-20", "21-100", "100+"])
    print("\nshare of coupons ever redeemed, by number of products the coupon covers:")
    print(cp.groupby("breadth", observed=True).agg(coupons=("redeemed", "size"), share_redeemed=("redeemed", "mean"),
                                                   mean_redemptions=("redemptions", "mean")).round(3).to_string())
    cp["reused"] = cp.campaigns > 1
    print("\nreused across campaigns vs single-campaign coupons:")
    print(cp.groupby("reused").agg(coupons=("redeemed", "size"), share_redeemed=("redeemed", "mean"),
                                   mean_redemptions=("redemptions", "mean")).round(3).to_string())
    top = cp.redemptions.sort_values(ascending=False)
    print(f"\ntop 10% of coupons ({int(.1 * len(cp))}) account for {top.iloc[:int(.1 * len(cp))].sum() / top.sum():.0%} of redemptions")


In [ ]:
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    hdr("2.7 WHEN DO HOUSEHOLDS REDEEM?")
    red["dur"] = red.END_DAY - red.START_DAY + 1
    red["day_in"] = red.DAY - red.START_DAY
    red["frac_in"] = red.day_in / red.dur
    first = red.sort_values("DAY").drop_duplicates(["household_key", "CAMPAIGN"])
    for lab, x in [("all redemptions", red), ("first redemption per household-campaign", first)]:
        q = x.frac_in.quantile([.25, .5, .75])
        print(f"{lab}: median day {x.day_in.median():.0f}; window elapsed at 25/50/75%: {q.iloc[0]:.0%} / {q.iloc[1]:.0%} / {q.iloc[2]:.0%}")
    print("\ncumulative share of first redemptions by share of window elapsed:")
    print(pd.Series({f"{int(p * 100)}% of window": f"{(first.frac_in <= p).mean():.1%}" for p in (.25, .5, .75, 1.0)}).to_string())
    print("\nby campaign type (first redemptions):")
    print(first.groupby("type").agg(n=("day_in", "size"), median_day=("day_in", "median"),
          median_share_of_window=("frac_in", "median"), median_window_days=("dur", "median")).round(2).to_string())
    same_day = red.groupby(["household_key", "CAMPAIGN"]).DAY.nunique()
    print(f"\nhousehold-campaigns where all redemptions happen on ONE shopping day: {(same_day == 1).mean():.0%}")

In [ ]:
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    hdr("2.8 CONCENTRATION AND REPEAT REDEMPTION")
    per_hh = red.groupby("household_key").size().sort_values(ascending=False)
    recipients = table.household_key.nunique()
    print(f"redeeming households: {len(per_hh)} of {recipients:,} recipients ({len(per_hh) / recipients:.1%})")
    for p in (.05, .10, .20):
        k = int(round(p * len(per_hh)))
        print(f"  top {p:.0%} of redeemers ({k}) make {per_hh.iloc[:k].sum() / per_hh.sum():.0%} of redemptions")

In [ ]:
coupon_efficiency = cp.groupby("breadth", observed=True).agg(
    coupons=("redeemed", "size"),
    share_ever_redeemed=("redeemed", "mean"),
    mean_redemptions=("redemptions", "mean"),
)
coupon_reuse = cp.groupby("reused").agg(
    coupon_codes=("redeemed", "size"),
    share_ever_redeemed=("redeemed", "mean"),
    mean_redemptions=("redemptions", "mean"),
).rename(index={False: "Single campaign", True: "Reused across campaigns"})
timing_by_type = first.groupby("type").agg(
    first_redemptions=("day_in", "size"),
    median_day=("day_in", "median"),
    median_share_of_window=("frac_in", "median"),
    median_window_days=("dur", "median"),
)

In [ ]:
never_redeemed = int((~cp.redeemed).sum())
ever_redeemed = int(cp.redeemed.sum())
coupon_top_10_share = top.iloc[:int(0.1 * len(cp))].sum() / top.sum()
household_top_10_count = max(1, int(round(0.1 * len(per_hh))))
household_top_10_share = per_hh.iloc[:household_top_10_count].sum() / per_hh.sum()
first_half_share = (first.frac_in <= 0.5).mean()

# Redemption by documented campaign type. Coupon-code utilisation is counted at
# campaign–coupon level because the same code can appear in more than one campaign.
type_assignments = table[["household_key", "CAMPAIGN", "type"]].drop_duplicates()
redeemed_assignment_pairs = red[["household_key", "CAMPAIGN"]].drop_duplicates().assign(
    redeemed_household_campaign=1
)
type_household_redemption = type_assignments.merge(
    redeemed_assignment_pairs, on=["household_key", "CAMPAIGN"], how="left"
)
type_household_redemption["redeemed_household_campaign"] = (
    type_household_redemption.redeemed_household_campaign.fillna(0)
)
type_household_redemption = type_household_redemption.groupby("type").agg(
    household_campaign_assignments=("redeemed_household_campaign", "size"),
    redeemed_household_campaigns=("redeemed_household_campaign", "sum"),
    household_campaign_redemption_rate=("redeemed_household_campaign", "mean"),
)

campaign_coupon_codes = coupon[["CAMPAIGN", "COUPON_UPC"]].drop_duplicates().merge(
    desc[["CAMPAIGN", "type"]], on="CAMPAIGN", how="left", validate="many_to_one"
)
redeemed_campaign_coupon_codes = red[["CAMPAIGN", "COUPON_UPC"]].drop_duplicates().assign(
    observed_redemption=1
)
campaign_coupon_codes = campaign_coupon_codes.merge(
    redeemed_campaign_coupon_codes, on=["CAMPAIGN", "COUPON_UPC"], how="left"
)
campaign_coupon_codes["observed_redemption"] = campaign_coupon_codes.observed_redemption.fillna(0)
type_coupon_redemption = campaign_coupon_codes.groupby("type").agg(
    listed_campaign_coupon_codes=("observed_redemption", "size"),
    codes_with_observed_redemption=("observed_redemption", "sum"),
    share_with_no_observed_redemption=("observed_redemption", lambda values: 1 - values.mean()),
)

redemption_by_campaign_type = (
    desc.groupby("type").CAMPAIGN.nunique().rename("campaigns").to_frame()
    .join(type_household_redemption)
    .join(type_coupon_redemption)
    .reindex(["TypeA", "TypeB", "TypeC"])
    .reset_index().rename(columns={
        "type": "Campaign type",
        "campaigns": "Campaigns",
        "household_campaign_assignments": "Household–campaign assignments",
        "redeemed_household_campaigns": "Assignments with at least one redemption",
        "household_campaign_redemption_rate": "Household–campaign redemption rate",
        "listed_campaign_coupon_codes": "Listed campaign–coupon codes",
        "codes_with_observed_redemption": "Codes with observed redemption",
        "share_with_no_observed_redemption": "Codes with no observed redemption",
    })
)
redemption_by_campaign_type["Coupon-code interpretation"] = [
    "Pool-level only: each household received 16 unknown selections",
    "Issued-bundle utilisation: every recipient received every listed code",
    "Issued-bundle utilisation: every recipient received every listed code",
]
redemption_by_campaign_type.insert(
    redemption_by_campaign_type.columns.get_loc("Assignments with at least one redemption") + 1,
    "Assignments with no redemption",
    redemption_by_campaign_type["Household–campaign assignments"]
    - redemption_by_campaign_type["Assignments with at least one redemption"],
)

evidence_table = pd.DataFrame({
    "Finding": [
        "Listed coupon codes with no observed redemption (all types pooled)",
        "Redemptions generated by top 10% of coupon codes",
        "Redemptions generated by top 10% of redeeming households",
        "First redemptions occurring within first half of campaign window",
    ],
    "Evidence": [
        f"{never_redeemed:,} of {len(cp):,} ({never_redeemed / len(cp):.1%})",
        f"{coupon_top_10_share:.1%}",
        f"{household_top_10_share:.1%}",
        f"{first_half_share:.1%}",
    ],
})
redemption_frequency = pd.cut(
    cp.redemptions,
    bins=[-1, 0, 1, 4, 9, np.inf],
    labels=["Never redeemed", "Redeemed exactly once", "Redeemed 2–4 times", "Redeemed 5–9 times", "Redeemed 10+ times"],
).value_counts(sort=False)
coupon_utilisation_table = redemption_frequency.rename("Coupon codes").to_frame()
coupon_utilisation_table["Share of coupon codes"] = (
    coupon_utilisation_table["Coupon codes"] / len(cp)
)
coupon_utilisation_table.loc["Total"] = [len(cp), 1.0]

coupon_concentration_rows = []
household_concentration_rows = []
for share in [0.05, 0.10, 0.20]:
    coupon_count = max(1, int(round(share * len(top))))
    household_count = max(1, int(round(share * len(per_hh))))
    coupon_concentration_rows.append({
        "Highest-redemption coupon codes": f"Top {share:.0%}",
        "Number of coupon codes": coupon_count,
        "Share of all redemptions generated": top.iloc[:coupon_count].sum() / top.sum(),
    })
    household_concentration_rows.append({
        "Most active redeeming households": f"Top {share:.0%}",
        "Number of households": household_count,
        "Share of all redemptions generated": per_hh.iloc[:household_count].sum() / per_hh.sum(),
    })
coupon_concentration_table = pd.DataFrame(coupon_concentration_rows)
household_concentration_table = pd.DataFrame(household_concentration_rows)
concentration_comparison = pd.DataFrame({
    "Highest-activity group": coupon_concentration_table["Highest-redemption coupon codes"],
    "Coupon codes in group": coupon_concentration_table["Number of coupon codes"],
    "Redemptions from those coupon codes": coupon_concentration_table["Share of all redemptions generated"],
    "Redeeming households in group": household_concentration_table["Number of households"],
    "Redemptions from those households": household_concentration_table["Share of all redemptions generated"],
})

redemption_timing_table = pd.DataFrame({
    "Campaign window elapsed": ["First 25%", "First 50%", "First 75%", "Full window"],
    "First redemptions completed": [
        (first.frac_in <= 0.25).mean(),
        (first.frac_in <= 0.50).mean(),
        (first.frac_in <= 0.75).mean(),
        (first.frac_in <= 1.00).mean(),
    ],
})

display(redemption_by_campaign_type.style.hide(axis="index").format({
    "Campaigns": "{:,.0f}",
    "Household–campaign assignments": "{:,.0f}",
    "Assignments with at least one redemption": "{:,.0f}",
    "Assignments with no redemption": "{:,.0f}",
    "Household–campaign redemption rate": "{:.1%}",
    "Listed campaign–coupon codes": "{:,.0f}",
    "Codes with observed redemption": "{:,.0f}",
    "Codes with no observed redemption": "{:.1%}",
}).set_properties(subset=["Coupon-code interpretation"], **{
    "text-align": "left", "white-space": "normal", "max-width": "340px"
}).set_caption(
    "Redemption by campaign type: household response and listed coupon-code utilisation"
))
display(evidence_table.style.hide(axis="index").set_properties(
    subset=["Finding", "Evidence"], **{"text-align": "left", "white-space": "normal"},
).set_caption("Headline findings from past coupon campaigns"))
display(coupon_utilisation_table.style.format({
    "Coupon codes": "{:,.0f}", "Share of coupon codes": "{:.1%}",
}).set_caption("Coupon-code utilisation frequency"))
display(concentration_comparison.style.hide(axis="index").format({
    "Coupon codes in group": "{:,.0f}",
    "Redemptions from those coupon codes": "{:.1%}",
    "Redeeming households in group": "{:,.0f}",
    "Redemptions from those households": "{:.1%}",
}).set_caption("Redemption concentration across coupon codes and redeeming households"))
display(redemption_timing_table.style.hide(axis="index").format({
    "First redemptions completed": "{:.1%}",
}).set_caption("Timing of first redemption within the campaign window"))

## Opportunity

### Type A

Retain the existing personalised approach—not because it is necessarily optimal, but because the dataset does not show which 16 coupons each household received. Consequently, household-level Type A coupon selection cannot be evaluated or redesigned fairly: a coupon that was not redeemed may never have been included among that household's 16 coupons.

### Types B/C

Move from universal complete bundles toward behavioural-segment targeting. The available data identifies every coupon included in a Type B/C campaign and every assigned household, making it possible to evaluate which customer segments appear valuable and campaign-responsive.

## 3. Behavioral segmentation

Households with at least five baskets are described by spend frequency, basket frequency, average basket size, recency, and discount share. Skewed variables are log-transformed and all variables are standardized.

The segmentation follows a **Ward → k-means hybrid procedure**:

1. Ward's hierarchical clustering builds the household hierarchy.
2. The dendrogram and scree/elbow evidence help select the number of groups.
3. After cutting the Ward tree, the mean coordinates of its groups initialise k-means.
4. K-means repeatedly reallocates households and updates the centroids until convergence.

Candidate solutions from 2–7 clusters are assessed using:

- explained variance (`R²`) and the elbow in marginal gains,
- repeated half-sample stability, and
- the size and business interpretability of the resulting profiles.

The four business labels are assigned from profile rules rather than raw cluster numbers, because cluster numbers can change between fits. A companion coupon-profile table then describes campaign engagement, use intensity and the coupon commodities that can be matched to actual purchases; it is kept separate from the shopping table so both remain readable.

In [ ]:
clustering_variable_guide = pd.DataFrame([
    {
        "Variable": "Spending per active week",
        "How it is calculated": "Total non-fuel sales value ÷ number of weeks containing a purchase",
        "Why it is included": "Captures the household's typical monetary value when actively shopping",
        "Interpretation caveat": "Uses purchase weeks rather than every calendar week, so prolonged inactivity is handled mainly by recency",
    },
    {
        "Variable": "Shopping trips per active week",
        "How it is calculated": "Number of baskets ÷ number of weeks containing a purchase",
        "Why it is included": "Separates frequent top-up shoppers from households making fewer shopping trips",
        "Interpretation caveat": "Multiple baskets may occasionally represent split transactions rather than separate physical visits",
    },
    {
        "Variable": "Average basket value",
        "How it is calculated": "Mean recorded sales value across the household's baskets",
        "Why it is included": "Distinguishes large stock-up missions from small convenience or top-up baskets",
        "Interpretation caveat": "Basket value is affected by product mix and prices as well as quantity purchased",
    },
    {
        "Variable": "Recency",
        "How it is calculated": "Dataset end day − household's most recent purchase day",
        "Why it is included": "Distinguishes currently active households from potentially lapsed households with similar historical spending",
        "Interpretation caveat": "High recency may reflect panel exit or incomplete observation rather than genuine churn",
    },
    {
        "Variable": "Retail-discount share",
        "How it is calculated": "Retail discounts ÷ estimated value before retail discounts",
        "Why it is included": "Shows how strongly the household's shopping behaviour is associated with discounted products, helping distinguish deal-oriented shoppers for coupon targeting",
        "Interpretation caveat": "Also reflects the promotions and categories available to the household; it is not a pure measure of price sensitivity",
    },
])
display(
    clustering_variable_guide.style.hide(axis="index")
    .set_properties(subset=["Variable", "How it is calculated", "Why it is included", "Interpretation caveat"],
                    **{"text-align": "left", "white-space": "normal", "max-width": "330px"})
    .set_caption(
        "Clustering variables and rationale. All are derived from transactions for 2,474 eligible households; "
        "demographics and coupon redemptions are attached only after clustering to avoid missing-data loss and circular segment definitions."
    )
)

In [ ]:
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    hdr("3. SEGMENTATION (RFM-style bases, fuel excluded)")
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans
    from sklearn.metrics import adjusted_rand_score
    from scipy.cluster.hierarchy import linkage, fcluster
    bk = shop.groupby(["household_key", "BASKET_ID"]).SALES_VALUE.sum()
    H = pd.DataFrame({"spend": shop.groupby("household_key").SALES_VALUE.sum(),
                      "weeks": shop.groupby("household_key").WEEK_NO.nunique(),
                      "baskets": bk.groupby("household_key").size(),
                      "avg_basket": bk.groupby("household_key").mean()})
    H["spend_per_week"] = H.spend / H.weeks
    H["baskets_per_week"] = H.baskets / H.weeks
    H["recency"] = t.DAY.max() - shop.groupby("household_key").DAY.max()
    s = shop.groupby("household_key")[["SALES_VALUE", "RETAIL_DISC"]].sum()
    H["disc_share"] = -s.RETAIL_DISC / (s.SALES_VALUE - s.RETAIL_DISC)
    H = H[H.baskets >= 5]
    cols = ["spend_per_week", "baskets_per_week", "avg_basket", "recency", "disc_share"]
    Xr = H[cols].copy()
    for c in cols[:4]: Xr[c] = np.log1p(Xr[c])                 # skewed -> log, then z-score (course M1)
    X = StandardScaler().fit_transform(Xr)
    tss = ((X - X.mean(0)) ** 2).sum()
    r2 = lambda lab: 1 - sum(((X[lab == j] - X[lab == j].mean(0)) ** 2).sum() for j in np.unique(lab)) / tss

    def ward_seeded_kmeans(data, k, ward_tree=None):
        """Cut a Ward tree at k, then use those group means to initialise K-means."""
        if ward_tree is None:
            ward_tree = linkage(data, method="ward")
        ward_labels = fcluster(ward_tree, t=k, criterion="maxclust") - 1
        ward_centroids = np.vstack([
            data[ward_labels == group].mean(axis=0) for group in range(k)
        ])
        model = KMeans(
            n_clusters=k, init=ward_centroids, n_init=1, random_state=0
        ).fit(data)
        return model, ward_labels, ward_tree

    ward_linkage = linkage(X, method="ward")
    print(f"households {len(H)}\n k  Ward_R2  KMeans_R2  ARI(Ward,KMeans)")
    for k in range(2, 8):
        km, w, _ = ward_seeded_kmeans(X, k, ward_linkage)
        print(f" {k}  {r2(w):.3f}    {r2(km.labels_):.3f}      "
              f"{adjusted_rand_score(w, km.labels_):.3f}")
    base, ward4_labels, _ = ward_seeded_kmeans(X, 4, ward_linkage)
    rng = np.random.default_rng(0); aris = []
    for i in range(20):
        idx = rng.choice(len(X), len(X) // 2, replace=False)
        sample_model, _, _ = ward_seeded_kmeans(X[idx], 4)
        aris.append(adjusted_rand_score(base.labels_[idx], sample_model.labels_))
    print(f"k=4 stability on 20 random halves: ARI mean {np.mean(aris):.3f}, min {np.min(aris):.3f}")
    H["seg"] = base.labels_
    prof = H.groupby("seg")[cols].median().round(3)
    prof["households"] = H.seg.value_counts(); prof["spend_share"] = (H.groupby("seg").spend.sum() / H.spend.sum()).round(3)
    # working names, assigned by rule so they survive label renumbering
    NAMES = {prof.avg_basket.idxmax(): "Big-basket regulars", prof.baskets_per_week.idxmax(): "Frequent top-up",
             prof.disc_share.idxmax(): "Deal seekers", prof.spend_per_week.idxmin(): "Light shoppers"}
    assert len(NAMES) == 4, "segment naming rule clashed - check profiles"
    SEG_ORDER = list(prof.spend_share.sort_values(ascending=False).index)   # largest share of spend first
    print(prof.rename(index=NAMES).loc[[NAMES[s] for s in SEG_ORDER]].to_string())

In [ ]:
segment_profile = prof.rename(index=NAMES).loc[[NAMES[s] for s in SEG_ORDER]]
segmentation_diagnostics = pd.DataFrame({
    "Measure": [
        "Households retained",
        "Selected clusters",
        "Ward-initialised K-means explained variance (R²)",
        "Half-sample stability: mean score",
        "Half-sample stability: lower-quartile score",
    ],
    "Result": [
        f"{len(H):,}",
        "4",
        f"{r2(base.labels_):.3f}",
        f"{np.mean(aris):.3f}",
        f"{np.quantile(aris, 0.25):.3f}",
    ],
})
display(segmentation_diagnostics.style.hide(axis="index").set_caption(
    "Selected four-cluster solution"
))

# Show the Ward hierarchy used to select and initialise the final solution.
from scipy.cluster.hierarchy import dendrogram
ward_cut_height = (ward_linkage[-4, 2] + ward_linkage[-3, 2]) / 2
fig, ax = plt.subplots(figsize=(14, 5.2), dpi=130)
dendrogram(
    ward_linkage,
    truncate_mode="lastp",
    p=30,
    show_leaf_counts=True,
    leaf_rotation=0,
    leaf_font_size=8,
    color_threshold=ward_cut_height,
    above_threshold_color="#9CA3AF",
    ax=ax,
)
ax.axhline(ward_cut_height, color="#10B981", linestyle="--", linewidth=2,
           label="Cut producing four Ward groups")
ax.set_title("Ward dendrogram: cut the hierarchy to obtain four initial groups", weight="bold")
ax.set_xlabel("Condensed household groups (numbers show households in each branch)")
ax.set_ylabel("Ward linkage distance")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

# Compare fit, elbow gains and resampling stability across candidate k values.
cluster_selection_rows = []
selection_rng = np.random.default_rng(4211)
# Use the same resamples for every candidate k so stability comparisons are fair.
selection_samples = [
    selection_rng.choice(len(X), len(X) // 2, replace=False)
    for _ in range(20)
]
selection_sample_trees = [
    linkage(X[sample_idx], method="ward") for sample_idx in selection_samples
]
for candidate_k in range(2, 8):
    candidate_kmeans, ward_labels, _ = ward_seeded_kmeans(
        X, candidate_k, ward_linkage
    )
    candidate_labels = candidate_kmeans.labels_

    candidate_stability = []
    for sample_idx, sample_tree in zip(selection_samples, selection_sample_trees):
        sample_model, _, _ = ward_seeded_kmeans(
            X[sample_idx], candidate_k, sample_tree
        )
        sample_labels = sample_model.labels_
        candidate_stability.append(
            adjusted_rand_score(candidate_labels[sample_idx], sample_labels)
        )

    cluster_selection_rows.append({
        "Clusters": candidate_k,
        "Ward R²": r2(ward_labels),
        "Ward-initialised K-means R²": r2(candidate_labels),
        "Mean stability": np.mean(candidate_stability),
        "Lower-quartile stability": np.quantile(candidate_stability, 0.25),
    })

cluster_selection = pd.DataFrame(cluster_selection_rows)
cluster_selection["Marginal refined R² gain"] = cluster_selection["Ward-initialised K-means R²"].diff()
cluster_selection.loc[cluster_selection.index[0], "Marginal refined R² gain"] = (
    cluster_selection.loc[cluster_selection.index[0], "Ward-initialised K-means R²"]
)
cluster_selection["Selected"] = np.where(cluster_selection.Clusters == 4, "Selected", "")

display(cluster_selection.style.hide(axis="index").format({
    "Ward R²": "{:.3f}",
    "Ward-initialised K-means R²": "{:.3f}",
    "Marginal refined R² gain": "+{:.3f}",
    "Mean stability": "{:.3f}",
    "Lower-quartile stability": "{:.3f}",
}).set_caption("Cluster-number comparison"))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), dpi=130)
k_values = cluster_selection.Clusters

axes[0].plot(k_values, cluster_selection["Ward R²"], marker="o", linewidth=2.2,
             color="#F59E0B", label="Ward")
axes[0].plot(k_values, cluster_selection["Ward-initialised K-means R²"], marker="o", linewidth=2.2,
             color="#2563EB", label="Ward-initialised K-means")
axes[0].axvline(4, color="#10B981", linestyle="--", linewidth=1.8)
axes[0].scatter([4], [cluster_selection.loc[cluster_selection.Clusters == 4, "Ward-initialised K-means R²"].iloc[0]],
                s=80, color="#10B981", zorder=4)
axes[0].set_title("A. Explained variation increases with k", weight="bold")
axes[0].set_xlabel("Number of clusters")
axes[0].set_ylabel("R²")
axes[0].set_xticks(k_values)
axes[0].legend(frameon=False)

gain_colours = ["#93C5FD" if k != 4 else "#10B981" for k in k_values]
bars = axes[1].bar(k_values, cluster_selection["Marginal refined R² gain"],
                   color=gain_colours, width=0.65)
axes[1].bar_label(
    bars,
    labels=[f"+{value:.3f}" for value in cluster_selection["Marginal refined R² gain"]],
    padding=3, fontsize=9,
)
axes[1].set_title("B. Marginal gain begins to flatten", weight="bold")
axes[1].set_xlabel("Number of clusters")
axes[1].set_ylabel("Increase in refined R²")
axes[1].set_xticks(k_values)

axes[2].plot(k_values, cluster_selection["Mean stability"], marker="o", linewidth=2.2,
             color="#7C3AED", label="Mean stability")
axes[2].plot(k_values, cluster_selection["Lower-quartile stability"], marker="o", linewidth=2.0,
             color="#DC2626", label="Lower-quartile stability")
axes[2].axvline(4, color="#10B981", linestyle="--", linewidth=1.8)
axes[2].set_ylim(0, 1)
axes[2].set_title("C. Stability generally weakens beyond four", weight="bold")
axes[2].set_xlabel("Number of clusters")
axes[2].set_ylabel("Resampling stability score (0–1)")
axes[2].set_xticks(k_values)
axes[2].legend(frameon=False, fontsize=9)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Four clusters balance fit, stability and business interpretability",
             fontsize=16, weight="bold", y=1.04)
fig.text(
    0.5, -0.02,
    "The elbow is not a mechanical rule: four clusters are retained because the next split adds less fit and is much less stable.",
    ha="center", fontsize=9, color="#4B5563",
)
plt.tight_layout()
plt.show()

display(segment_profile.style.format({
    "spend_per_week": "${:,.2f}",
    "baskets_per_week": "{:.2f}",
    "avg_basket": "${:,.2f}",
    "recency": "{:.0f} days",
    "disc_share": "{:.1%}",
    "households": "{:,.0f}",
    "spend_share": "{:.1%}",
}).set_caption("Median behavioural profile by customer segment"))

# Presentation summary: behaviour, selected over-indexing commodities and demographics.
segment_names = H.seg.map(NAMES)
segment_transactions = shop[["household_key", "COMMODITY_DESC", "SALES_VALUE"]].merge(
    segment_names.rename("segment"), left_on="household_key", right_index=True, how="inner"
)
commodity_sales = segment_transactions.groupby(["segment", "COMMODITY_DESC"]).SALES_VALUE.sum().unstack(fill_value=0)
commodity_share = commodity_sales.div(commodity_sales.sum(axis=1), axis=0)
panel_commodity_share = (
    segment_transactions.groupby("COMMODITY_DESC").SALES_VALUE.sum()
    / segment_transactions.SALES_VALUE.sum()
)
commodity_index = commodity_share.div(panel_commodity_share, axis=1)

preference_specs = {
    "Big-basket regulars": [
        ("Cheeses", "CHEESES"),
        ("Household cleaning", "HOUSEHOLD CLEANG NEEDS"),
        ("Cereal/breakfast", "CEREAL/BREAKFAST"),
    ],
    "Frequent top-up": [
        ("Cigarettes", "CIGARETTES"),
        ("Domestic wine", "DOMESTIC WINE"),
        ("Soft drinks", "SOFT DRINKS"),
        ("Diapers", "DIAPERS & DISPOSABLES"),
    ],
    "Deal seekers": [
        ("Ice cream", "ICE CREAM/MILK/SHERBTS"),
        ("Pork", "PORK"),
        ("Dinner sausage", "DINNER SAUSAGE"),
        ("Soft drinks", "SOFT DRINKS"),
    ],
    "Light shoppers": [
        ("Cakes", "CAKES"),
        ("Beer", "BEERS/ALES"),
        ("Juice", "JUICE"),
    ],
}
preference_text = {
    segment: ", ".join(
        f"{label} ({commodity_index.loc[segment, commodity]:.2f}×)"
        for label, commodity in choices
    )
    for segment, choices in preference_specs.items()
}
assert all(
    commodity_index.loc[segment, commodity] > 1
    for segment, choices in preference_specs.items()
    for _, commodity in choices
), "Selected preferences must over-index relative to the panel"

demographic_segments = demo.merge(
    segment_names.rename("segment"), left_on="household_key", right_index=True, how="inner"
)
demographic_counts = demographic_segments.segment.value_counts()

def segment_demo_share(segment, column, accepted_values):
    values = demographic_segments.loc[demographic_segments.segment == segment, column]
    return values.isin(accepted_values).mean()

demographic_tendency = {
    "Big-basket regulars": (
        f"Two-adult households ({segment_demo_share('Big-basket regulars', 'HH_COMP_DESC', ['2 Adults No Kids', '2 Adults Kids']):.0%}); "
        f"$50–99K income ({segment_demo_share('Big-basket regulars', 'INCOME_DESC', ['50-74K', '75-99K']):.0%})"
    ),
    "Frequent top-up": (
        f"One-person households ({segment_demo_share('Frequent top-up', 'HOUSEHOLD_SIZE_DESC', ['1']):.0%})"
    ),
    "Deal seekers": (
        f"Households with children ({segment_demo_share('Deal seekers', 'KID_CATEGORY_DESC', ['1', '2', '3+']):.0%}); "
        f"4+ members ({segment_demo_share('Deal seekers', 'HOUSEHOLD_SIZE_DESC', ['4', '5+']):.0%})"
    ),
    "Light shoppers": (
        f"One-person households ({segment_demo_share('Light shoppers', 'HOUSEHOLD_SIZE_DESC', ['1']):.0%})"
    ),
}

# Attach coupon behaviour to the segments. Exact coupon categories are based on
# redemptions that can be matched to the coupon-discounted product bought that day.
coupon_product_category = coupon.merge(
    prod[["PRODUCT_ID", "DEPARTMENT", "COMMODITY_DESC"]],
    on="PRODUCT_ID", how="left", validate="many_to_one",
)
coupon_scope = coupon_product_category.groupby(["COUPON_UPC", "CAMPAIGN"]).agg(
    eligible_products=("PRODUCT_ID", "nunique"),
    departments=("DEPARTMENT", "nunique"),
    commodities=("COMMODITY_DESC", "nunique"),
).reset_index()
coupon_scope["scope"] = np.where(
    coupon_scope.departments == 1, "single department", "multiple departments"
)
coupon_transaction_lines = t.loc[
    t.COUPON_DISC < 0,
    ["household_key", "DAY", "PRODUCT_ID", "COUPON_DISC", "COUPON_MATCH_DISC", "DEPARTMENT"],
]
redemption_items = (
    red.merge(
        coupon[["COUPON_UPC", "CAMPAIGN", "PRODUCT_ID"]],
        on=["COUPON_UPC", "CAMPAIGN"], how="inner",
    )
    .merge(
        coupon_transaction_lines,
        on=["household_key", "DAY", "PRODUCT_ID"], how="inner",
    )
    .drop_duplicates(["household_key", "DAY", "COUPON_UPC", "CAMPAIGN"])
    .merge(
        prod[["PRODUCT_ID", "COMMODITY_DESC"]],
        on="PRODUCT_ID", how="left", validate="many_to_one",
    )
)
redemption_items["seg"] = redemption_items.household_key.map(H.seg)
redemption_items["segment"] = redemption_items.seg.map(NAMES)

overview_rows = []
for segment_id in SEG_ORDER:
    segment = NAMES[segment_id]
    profile = segment_profile.loc[segment]
    count = int(profile.households)
    if segment == "Big-basket regulars":
        shopping = f"${profile.spend_per_week:.0f}/week; ${profile.avg_basket:.0f} average basket; {profile.spend_share:.0%} of panel spend"
    elif segment == "Frequent top-up":
        shopping = f"{profile.baskets_per_week:.1f} trips/week; ${profile.avg_basket:.0f} average basket; {profile.spend_share:.0%} of panel spend"
    elif segment == "Deal seekers":
        shopping = f"${profile.spend_per_week:.0f}/week; highest retail-discount share ({profile.disc_share:.1%})"
    else:
        shopping = f"${profile.spend_per_week:.0f}/week; ${profile.avg_basket:.0f} average basket; {profile.spend_share:.0%} of panel spend"
    overview_rows.append({
        "Segment": segment,
        "Households": f"{count:,} ({count / len(H):.0%})",
        "How they shop": shopping,
        "Selected commodities bought more than panel average": preference_text[segment],
        "Demographic tendency*": demographic_tendency[segment],
    })

segment_overview = pd.DataFrame(overview_rows)
display(
    segment_overview.style.hide(axis="index")
    .set_properties(subset=[
        "Segment", "How they shop", "Selected commodities bought more than panel average", "Demographic tendency*"
    ], **{"text-align": "left", "white-space": "normal", "max-width": "290px"})
    .set_caption(
        "Customer-segment overview. Commodity values in parentheses are spending-share indices (1.00× = panel average). "
        "*Demographic tendencies use 801 households: "
        + ", ".join(f"{name} n={demographic_counts.get(name, 0):,}" for name in [NAMES[s] for s in SEG_ORDER])
        + "."
    )
)

# A second, narrower table keeps the coupon evidence legible on a slide.
segment_pairs = pairs.loc[pairs.household_key.isin(H.index)].copy()
segment_pairs["seg"] = segment_pairs.household_key.map(H.seg)
coupon_engagement = segment_pairs.groupby("seg").agg(
    campaign_assignments=("CAMPAIGN", "size"),
    targeted_households=("household_key", "nunique"),
    redeemed_assignments=("y", "sum"),
    assignment_redemption_rate=("y", "mean"),
)
redeeming_households = segment_pairs.loc[segment_pairs.y == 1].groupby("seg").household_key.nunique()
coupon_engagement["targeted_households_ever_redeeming"] = (
    redeeming_households / coupon_engagement.targeted_households
)

raw_segment_redemptions = red.assign(seg=red.household_key.map(H.seg)).dropna(subset=["seg"])
redemption_intensity = raw_segment_redemptions.groupby("seg").size().div(
    raw_segment_redemptions.groupby("seg").household_key.nunique()
)
matched_support = redemption_items.groupby("seg").agg(
    matched_redemptions=("COUPON_UPC", "size"),
    retailer_match_share=("COUPON_MATCH_DISC", lambda values: values.lt(0).mean()),
)

commodity_counts = redemption_items.dropna(subset=["seg", "COMMODITY_DESC"]).groupby(
    ["seg", "COMMODITY_DESC"]
).size()
top_coupon_commodities = {}
for segment_id in SEG_ORDER:
    if segment_id in commodity_counts.index.get_level_values("seg"):
        top = commodity_counts.loc[segment_id].nlargest(3)
        top_coupon_commodities[segment_id] = ", ".join(
            commodity.title() for commodity in top.index
        )
    else:
        top_coupon_commodities[segment_id] = "No product-matched redemptions"

coupon_profile_rows = []
for segment_id in SEG_ORDER:
    engagement = coupon_engagement.loc[segment_id]
    segment_households = int(segment_profile.loc[NAMES[segment_id], "households"])
    coupon_profile_rows.append({
        "Segment": NAMES[segment_id],
        "Targeted households": (
            f"{int(engagement.targeted_households):,} "
            f"({engagement.targeted_households / segment_households:.0%} of segment)"
        ),
        "Coupon engagement": (
            f"{engagement.assignment_redemption_rate:.1%} of campaign assignments redeemed; "
            f"{engagement.targeted_households_ever_redeeming:.1%} of targeted households ever redeemed"
        ),
        "Use among redeemers": f"{redemption_intensity.get(segment_id, 0):.1f} redemptions per redeeming household",
        "Most-used matched coupon commodities*": top_coupon_commodities[segment_id],
        "Retailer match support*": f"{matched_support.loc[segment_id, 'retailer_match_share']:.1%} of matched redemptions",
    })

segment_coupon_profile = pd.DataFrame(coupon_profile_rows)
display(
    segment_coupon_profile.style.hide(axis="index")
    .set_properties(subset=[
        "Segment", "Coupon engagement", "Use among redeemers",
        "Most-used matched coupon commodities*", "Retailer match support*",
    ], **{"text-align": "left", "white-space": "normal", "max-width": "300px"})
    .set_caption(
        "Coupon behaviour by customer segment. Assignment redemption means at least one coupon was redeemed "
        "from a targeted campaign. *Commodity and retailer-match results use only the "
        f"{len(redemption_items):,} of {len(red):,} redemptions ({len(redemption_items) / len(red):.1%}) "
        "that could be matched to the purchased product. Commodity rankings reflect both which coupons were offered "
        "and which were redeemed, rather than unaided category preference. Retailer match means a recorded "
        "coupon-match discount and does not apply to every coupon."
    )
)

## 4. Proposal: greater selectivity in Type B/C campaign targeting

### Current Type B/C approach

Every household assigned to a Type B or C campaign receives the same complete coupon bundle.

### Proposed revision

Divide campaign recipients into behavioural segments and decide more selectively which segments should receive Type B/C campaigns.

### Why the proposal focuses on audiences rather than prescribing exact bundles

We first tested whether particular product categories could be matched to each behavioural segment using two measures:

1. **Past-purchase affinity:** whether the segment normally bought more from a category than the average household.
2. **Coupon-linked spending response:** whether households receiving campaigns containing that category increased their spending more than similar households that did not receive the campaign.

Department-level findings were too broad to create a meaningful change because Grocery and Drug GM were already widely included in existing Type B/C bundles. Commodity analysis produced more specific categories, but too few combinations had reliable observations and patterns learned from earlier campaigns did not remain stable later. The data therefore cannot reliably prescribe exact segment-specific coupon bundles.

The evidence is stronger at customer-segment level: CLV can identify valuable segments, while matched campaign lift can show whether their overall spending increases when they receive a campaign. Consequently, the main proposed improvement is to distribute Type B/C campaigns more selectively to valuable and campaign-responsive segments. Smaller commodity-level bundles remain controlled-pilot ideas.

The following subsections document the category feasibility checks supporting this conclusion. Type B/C coupon mappings are joined conservatively to departments and commodities; ambiguous multi-category redemptions remain unassigned rather than being forced into a category.

In [ ]:
# Type B/C coupon category coverage and redemption checks

BC_CAMPAIGNS = set(desc.loc[desc.type.isin(["TypeB", "TypeC"]), "CAMPAIGN"])
bc_coupon_product_category = coupon_product_category.loc[
    coupon_product_category.CAMPAIGN.isin(BC_CAMPAIGNS)
].copy()
bc_coupon_scope = coupon_scope.loc[coupon_scope.CAMPAIGN.isin(BC_CAMPAIGNS)].copy()
bc_redemption_items = redemption_items.loc[
    redemption_items.CAMPAIGN.isin(BC_CAMPAIGNS)
].copy()
bc_raw_redemptions = red.loc[red.CAMPAIGN.isin(BC_CAMPAIGNS)].copy()

# Map redemptions directly when a campaign–coupon pair belongs to one department.
# Multi-department coupons remain unassigned because the redemption file has no product ID.
bc_coupon_department_scope = bc_coupon_product_category.groupby(["CAMPAIGN", "COUPON_UPC"]).agg(
    department_count=("DEPARTMENT", "nunique"),
    DEPARTMENT=("DEPARTMENT", "first"),
).reset_index()
bc_single_department_coupon_map = bc_coupon_department_scope.loc[
    bc_coupon_department_scope.department_count == 1,
    ["CAMPAIGN", "COUPON_UPC", "DEPARTMENT"],
]
bc_category_redemption_events = bc_raw_redemptions.merge(
    bc_single_department_coupon_map,
    on=["CAMPAIGN", "COUPON_UPC"], how="inner", validate="many_to_one",
)

category_coverage = bc_coupon_product_category.groupby("DEPARTMENT").agg(
    campaigns=("CAMPAIGN", "nunique"),
    coupons=("COUPON_UPC", "nunique"),
    eligible_products=("PRODUCT_ID", "nunique"),
)
redemptions_by_category = bc_category_redemption_events.groupby("DEPARTMENT").size().rename(
    "single_department_redemptions"
)
category_coverage = category_coverage.join(redemptions_by_category).fillna({
    "single_department_redemptions": 0
})

# Include categories observed in at least two B/C campaigns with at least one
# unambiguous single-department redemption. Sparse categories remain exploratory.
TOP_COUPON_CATEGORIES = category_coverage.query(
    "single_department_redemptions >= 1 and campaigns >= 2"
).sort_values(["campaigns", "single_department_redemptions"], ascending=False).index.tolist()

category_mapping_summary = pd.DataFrame({
    "Measure": [
        "Single-department coupon–campaign pairs",
        "B/C redemptions assigned to one department",
        "Departments included in utilisation analysis",
    ],
    "Result": [
        f"{(bc_coupon_scope.scope == 'single department').mean():.1%}",
        f"{len(bc_category_redemption_events):,} of {len(bc_raw_redemptions):,} ({len(bc_category_redemption_events) / len(bc_raw_redemptions):.1%})",
        ", ".join(TOP_COUPON_CATEGORIES),
    ],
})
display(category_mapping_summary.style.hide(axis="index").set_caption("Type B/C coupon-category mapping checks"))
display(category_coverage.loc[TOP_COUPON_CATEGORIES].astype(int).style.format("{:,.0f}").set_caption(
    "B/C departments included in segment–category utilisation analysis"
))
display(bc_coupon_scope.groupby("scope").agg(
    coupon_campaign_pairs=("COUPON_UPC", "size"),
    median_products=("eligible_products", "median"),
    median_commodities=("commodities", "median"),
).style.format("{:,.1f}").set_caption("Type B/C coupon breadth by category scope"))

In [ ]:
# Redemption by segment and coupon department

# For Types B/C, a campaign assignment is known exposure to the complete coupon bundle.
# Category exposure uses departments represented by unambiguous single-department coupons.
campaign_categories = bc_single_department_coupon_map.loc[
    bc_single_department_coupon_map.DEPARTMENT.isin(TOP_COUPON_CATEGORIES),
    ["CAMPAIGN", "DEPARTMENT"],
].drop_duplicates()
category_exposure = table.loc[
    table.CAMPAIGN.isin(BC_CAMPAIGNS), ["household_key", "CAMPAIGN"]
].merge(
    campaign_categories, on="CAMPAIGN", how="inner"
)
category_exposure["seg"] = category_exposure.household_key.map(H.seg)
category_exposure = category_exposure.dropna(subset=["seg"])
category_exposure["seg"] = category_exposure.seg.astype(int)

category_redemption = bc_category_redemption_events.loc[
    bc_category_redemption_events.DEPARTMENT.isin(TOP_COUPON_CATEGORIES)
].groupby(["household_key", "CAMPAIGN", "DEPARTMENT"]).size().rename("redemptions")
category_exposure = category_exposure.merge(
    category_redemption,
    left_on=["household_key", "CAMPAIGN", "DEPARTMENT"],
    right_index=True,
    how="left",
)
category_exposure["redeemed"] = category_exposure.redemptions.fillna(0).gt(0)

segment_category_redemption = category_exposure.groupby(["seg", "DEPARTMENT"]).agg(
    household_campaign_exposures=("redeemed", "size"),
    matched_redeemers=("redeemed", "sum"),
    matched_redemption_rate=("redeemed", "mean"),
).reset_index()
segment_category_redemption["segment"] = segment_category_redemption.seg.map(NAMES)
segment_category_redemption = segment_category_redemption.pivot(
    index="segment", columns="DEPARTMENT", values="matched_redemption_rate"
).reindex([NAMES[s] for s in SEG_ORDER])
display(segment_category_redemption.style.format("{:.1%}").set_caption(
    "Type B/C category redemption rate by behavioural segment and department"
))

### 4.1 Test smaller segment-specific category bundles

Earlier Type B/C campaigns are used to rank coupon categories within each behavioural segment. A simple planning rule selects the smallest set of departments accounting for at least 80% of that segment's earlier observed category redemptions. The selected set is then evaluated on later campaigns that were not used to choose it.

This is a **coupon-utilisation test**, not an incremental-profit estimate. It asks whether fewer category exposures could retain most observed redemptions. Categories with very few campaigns or redemptions remain exploratory, and a controlled future campaign is required to test whether smaller bundles improve profit.

In [ ]:
# Earlier-to-later validation of segment-specific B/C category bundles.
category_exposure = category_exposure.merge(
    desc[["CAMPAIGN", "START_DAY"]], on="CAMPAIGN", how="left", validate="many_to_one"
)
category_exposure["period"] = np.where(
    category_exposure.START_DAY < 500, "Earlier campaigns", "Later campaigns"
)

segment_category_period_response = category_exposure.groupby(
    ["period", "seg", "DEPARTMENT"]
).agg(
    campaigns=("CAMPAIGN", "nunique"),
    category_exposures=("redeemed", "size"),
    redeemed_exposures=("redeemed", "sum"),
    redemption_rate=("redeemed", "mean"),
).reset_index()
segment_category_period_response["segment"] = segment_category_period_response.seg.map(NAMES)

earlier_response = segment_category_period_response.loc[
    segment_category_period_response.period == "Earlier campaigns"
].copy()
later_response = segment_category_period_response.loc[
    segment_category_period_response.period == "Later campaigns"
].copy()

# Select the smallest department set capturing at least 80% of earlier redemptions.
selection_rows = []
for segment_id in SEG_ORDER:
    frame = earlier_response.loc[earlier_response.seg == segment_id].sort_values(
        ["redeemed_exposures", "redemption_rate"], ascending=False
    ).copy()
    total_redemptions = frame.redeemed_exposures.sum()
    frame["cumulative_redemption_share"] = (
        frame.redeemed_exposures.cumsum() / total_redemptions if total_redemptions > 0 else 0
    )
    previous_share = frame.cumulative_redemption_share.shift(fill_value=0)
    frame["selected_for_80pct_bundle"] = previous_share < 0.80
    selection_rows.append(frame)
bundle_category_selection = pd.concat(selection_rows, ignore_index=True)

selected_pairs = set(
    bundle_category_selection.loc[
        bundle_category_selection.selected_for_80pct_bundle,
        ["seg", "DEPARTMENT"],
    ].itertuples(index=False, name=None)
)
later_test = category_exposure.loc[category_exposure.period == "Later campaigns"].copy()
later_test["selected_category"] = [
    (segment_id, department) in selected_pairs
    for segment_id, department in zip(later_test.seg, later_test.DEPARTMENT)
]

bundle_test_rows = []
for segment_id in SEG_ORDER:
    frame = later_test.loc[later_test.seg == segment_id]
    selected = frame.loc[frame.selected_category]
    all_redemptions = frame.redeemed.sum()
    selected_redemptions = selected.redeemed.sum()
    selected_categories = bundle_category_selection.loc[
        (bundle_category_selection.seg == segment_id)
        & bundle_category_selection.selected_for_80pct_bundle,
        "DEPARTMENT",
    ].tolist()
    bundle_test_rows.append({
        "Customer segment": NAMES[segment_id],
        "Categories selected from earlier campaigns": ", ".join(selected_categories),
        "Earlier categories observed": bundle_category_selection.loc[
            bundle_category_selection.seg == segment_id, "DEPARTMENT"
        ].nunique(),
        "Selected categories": len(selected_categories),
        "Later category exposures retained": len(selected) / len(frame) if len(frame) else np.nan,
        "Later observed redemptions retained": (
            selected_redemptions / all_redemptions if all_redemptions else np.nan
        ),
        "All-category later redemption rate": frame.redeemed.mean() if len(frame) else np.nan,
        "Selected-category later redemption rate": selected.redeemed.mean() if len(selected) else np.nan,
        "Later redemptions": int(all_redemptions),
    })
bundle_holdout_test = pd.DataFrame(bundle_test_rows)
selected_later_all = later_test.loc[later_test.selected_category]
bundle_holdout_test = pd.concat([
    bundle_holdout_test,
    pd.DataFrame([{
        "Customer segment": "Overall — segment-specific rule",
        "Categories selected from earlier campaigns": "Varies by segment",
        "Earlier categories observed": np.nan,
        "Selected categories": np.nan,
        "Later category exposures retained": len(selected_later_all) / len(later_test),
        "Later observed redemptions retained": (
            selected_later_all.redeemed.sum() / later_test.redeemed.sum()
        ),
        "All-category later redemption rate": later_test.redeemed.mean(),
        "Selected-category later redemption rate": selected_later_all.redeemed.mean(),
        "Later redemptions": int(later_test.redeemed.sum()),
    }]),
], ignore_index=True)

period_response_display = segment_category_period_response.pivot_table(
    index=["segment", "DEPARTMENT"], columns="period", values="redemption_rate"
).reset_index().rename(columns={
    "segment": "Customer segment",
    "DEPARTMENT": "Coupon department",
    "Earlier campaigns": "Earlier redemption rate",
    "Later campaigns": "Later redemption rate",
})
support_display = segment_category_period_response.pivot_table(
    index=["segment", "DEPARTMENT"], columns="period", values="campaigns"
).reset_index().rename(columns={
    "segment": "Customer segment",
    "DEPARTMENT": "Coupon department",
    "Earlier campaigns": "Earlier campaigns",
    "Later campaigns": "Later campaigns",
})
period_response_display = period_response_display.merge(
    support_display, on=["Customer segment", "Coupon department"], how="left"
).sort_values(["Customer segment", "Later redemption rate"], ascending=[True, False])

display(period_response_display.style.hide(axis="index").format({
    "Earlier redemption rate": "{:.1%}", "Later redemption rate": "{:.1%}",
    "Earlier campaigns": "{:,.0f}", "Later campaigns": "{:,.0f}",
}, na_rep="—").set_caption(
    "Type B/C segment–category redemption: earlier pattern and later-campaign check"
))
display(bundle_holdout_test.style.hide(axis="index").format({
    "Earlier categories observed": "{:,.0f}", "Selected categories": "{:,.0f}",
    "Later category exposures retained": "{:.1%}",
    "Later observed redemptions retained": "{:.1%}",
    "All-category later redemption rate": "{:.1%}",
    "Selected-category later redemption rate": "{:.1%}",
    "Later redemptions": "{:,.0f}",
}, na_rep="—").set_properties(subset=["Categories selected from earlier campaigns"], **{
    "text-align": "left", "white-space": "normal", "max-width": "320px"
}).set_caption(
    "Holdout test of an 80%-redemption category-bundle rule learned from earlier campaigns"
))

### 4.2 Combine category affinity with coupon-linked spending evidence

The first check measures **category affinity** using purchases before the first Type B/C campaign: an index above 1 means the segment devoted a larger share of spending to that department than households overall.

The second check estimates **coupon-linked category spending lift**. For every Type B/C campaign and department, recipients are compared with non-recipients from the same behavioural segment who had similar recent total and category spending. Positive lift in both the earlier and later campaign periods is stronger evidence than a one-period result.

The output deliberately uses the term **pilot candidate**, not final recommendation. Campaigns contain multiple coupons, assignment was not random, and some departments have limited campaign coverage. A controlled Type B/C bundle test is still needed to establish incremental value.

In [ ]:
# Two-check category screen: pre-campaign affinity + matched coupon-linked spending.

# 1) Category affinity before the first Type B/C campaign began.
first_bc_start = int(desc.loc[desc.CAMPAIGN.isin(BC_CAMPAIGNS), "START_DAY"].min())
affinity_transactions = shop.loc[shop.DAY < first_bc_start].copy()
affinity_transactions["seg"] = affinity_transactions.household_key.map(H.seg)
affinity_transactions = affinity_transactions.dropna(subset=["seg"])
affinity_transactions["seg"] = affinity_transactions.seg.astype(int)

segment_total_spend = affinity_transactions.groupby("seg").SALES_VALUE.sum()
overall_total_spend = affinity_transactions.SALES_VALUE.sum()
segment_department_spend = affinity_transactions.loc[
    affinity_transactions.DEPARTMENT.isin(TOP_COUPON_CATEGORIES)
].groupby(["seg", "DEPARTMENT"]).SALES_VALUE.sum()
overall_department_share = (
    affinity_transactions.loc[
        affinity_transactions.DEPARTMENT.isin(TOP_COUPON_CATEGORIES)
    ].groupby("DEPARTMENT").SALES_VALUE.sum() / overall_total_spend
)

affinity_grid = pd.MultiIndex.from_product(
    [SEG_ORDER, TOP_COUPON_CATEGORIES], names=["seg", "DEPARTMENT"]
).to_frame(index=False)
affinity_grid["segment_category_spend"] = [
    segment_department_spend.get((segment_id, department), 0)
    for segment_id, department in zip(affinity_grid.seg, affinity_grid.DEPARTMENT)
]
affinity_grid["segment_category_share"] = (
    affinity_grid.segment_category_spend / affinity_grid.seg.map(segment_total_spend)
)
affinity_grid["overall_category_share"] = affinity_grid.DEPARTMENT.map(overall_department_share)
affinity_grid["affinity_index"] = (
    affinity_grid.segment_category_share / affinity_grid.overall_category_share
)
affinity_grid["affinity_label"] = np.select(
    [affinity_grid.affinity_index >= 1.10, affinity_grid.affinity_index <= 0.90],
    ["Above-average affinity", "Below-average affinity"],
    default="About-average affinity",
)

# 2) Matched recipient-versus-non-recipient category spending comparison.
daily_total = shop.groupby(["household_key", "DAY"]).SALES_VALUE.sum().reset_index()
daily_department = shop.loc[
    shop.DEPARTMENT.isin(TOP_COUPON_CATEGORIES)
].groupby(["household_key", "DAY", "DEPARTMENT"]).SALES_VALUE.sum().reset_index()

matched_category_rows = []
for campaign_id, recipients in hc.loc[hc.CAMPAIGN.isin(BC_CAMPAIGNS)].groupby("CAMPAIGN"):
    start = int(recipients.START_DAY.iloc[0])
    end = int(recipients.END_DAY.iloc[0])
    duration = end - start + 1
    offered_departments = campaign_categories.loc[
        campaign_categories.CAMPAIGN == campaign_id, "DEPARTMENT"
    ].unique()

    def total_window(first_day, last_day):
        return daily_total.loc[
            daily_total.DAY.between(first_day, last_day)
        ].groupby("household_key").SALES_VALUE.sum().reindex(H.index).fillna(0)

    total_8w = total_window(start - 56, start - 1) / 56
    total_26w = total_window(start - 182, start - 1) / 182
    overall_bin = (
        pd.qcut(total_26w.rank(method="first"), 5, labels=False) * 3
        + pd.qcut(total_8w.rank(method="first"), 3, labels=False)
    ).astype(int)
    treated = pd.Series(H.index.isin(recipients.household_key), index=H.index)

    for department in offered_departments:
        department_daily = daily_department.loc[daily_department.DEPARTMENT == department]

        def department_window(first_day, last_day):
            return department_daily.loc[
                department_daily.DAY.between(first_day, last_day)
            ].groupby("household_key").SALES_VALUE.sum().reindex(H.index).fillna(0)

        department_8w = department_window(start - 56, start - 1) / 56
        department_26w = department_window(start - 182, start - 1) / 182
        department_during = department_window(start, end) / duration
        department_change = department_during - department_8w

        active = department_26w > 0
        category_bin = pd.Series(0, index=H.index, dtype=int)
        if active.sum() >= 3:
            category_bin.loc[active] = (
                pd.qcut(department_26w[active].rank(method="first"), 3, labels=False).astype(int) + 1
            )
        comparison = pd.DataFrame({
            "household_key": H.index,
            "treated": treated.values,
            "change": department_change.values,
            "baseline": department_8w.values,
            "match_bin": (overall_bin * 4 + category_bin).values,
            "seg": H.seg.values,
        })
        controls = comparison.loc[~comparison.treated].groupby(
            ["seg", "match_bin"]
        ).change.mean().rename("control_change").reset_index()
        treated_rows = comparison.loc[comparison.treated].merge(
            controls, on=["seg", "match_bin"], how="left"
        ).dropna(subset=["control_change"])

        for segment_id, segment_rows in treated_rows.groupby("seg"):
            matched_category_rows.append({
                "CAMPAIGN": campaign_id,
                "period": "Earlier" if start < 500 else "Later",
                "seg": int(segment_id),
                "DEPARTMENT": department,
                "recipients": len(segment_rows),
                "campaign_days": duration,
                "incremental_sales": (
                    (segment_rows.change - segment_rows.control_change).mean()
                    * len(segment_rows) * duration
                ),
                "baseline_sales": segment_rows.baseline.mean() * len(segment_rows) * duration,
            })

matched_category = pd.DataFrame(matched_category_rows)
matched_category_period = matched_category.groupby(
    ["period", "seg", "DEPARTMENT"]
).agg(
    campaigns=("CAMPAIGN", "nunique"),
    recipient_campaign_pairs=("recipients", "sum"),
    incremental_sales=("incremental_sales", "sum"),
    baseline_sales=("baseline_sales", "sum"),
).reset_index()
matched_category_period["lift_pct"] = np.where(
    matched_category_period.baseline_sales > 0,
    100 * matched_category_period.incremental_sales / matched_category_period.baseline_sales,
    np.nan,
)

lift_wide = matched_category_period.pivot_table(
    index=["seg", "DEPARTMENT"], columns="period",
    values=["campaigns", "lift_pct"], aggfunc="first"
)
lift_wide.columns = [f"{metric}_{period.lower()}" for metric, period in lift_wide.columns]
lift_wide = lift_wide.reset_index()

category_decision = affinity_grid.merge(
    lift_wide, on=["seg", "DEPARTMENT"], how="left"
)
for column in ["campaigns_earlier", "campaigns_later", "lift_pct_earlier", "lift_pct_later"]:
    if column not in category_decision:
        category_decision[column] = np.nan

enough_both = (
    category_decision.campaigns_earlier.ge(2)
    & category_decision.campaigns_later.ge(2)
)
positive_both = (
    category_decision.lift_pct_earlier.gt(0)
    & category_decision.lift_pct_later.gt(0)
)
positive_either = (
    category_decision.lift_pct_earlier.gt(0)
    | category_decision.lift_pct_later.gt(0)
)
category_decision["spending_evidence"] = np.select(
    [enough_both & positive_both, positive_both, positive_either],
    [
        "Positive in both periods",
        "Positive twice, but sparse",
        "Mixed across periods",
    ],
    default="No positive evidence or insufficient data",
)

stable_positive = enough_both & positive_both
repeated_but_sparse = ~enough_both & positive_both
category_decision["suggested_action"] = np.select(
    [
        stable_positive & category_decision.affinity_index.ge(0.90),
        stable_positive & category_decision.affinity_index.lt(0.90),
        repeated_but_sparse,
        positive_either & category_decision.affinity_index.ge(1.10),
        category_decision.affinity_index.ge(1.10),
    ],
    [
        "Higher-confidence controlled pilot",
        "Higher-confidence cross-sell pilot",
        "Limited-evidence pilot; repeat result but sparse",
        "Affinity-led exploratory test; spending evidence is mixed",
        "Relevant category, but do not discount automatically",
    ],
    default="Low priority until stronger evidence",
)
category_decision["segment"] = category_decision.seg.map(NAMES)
action_order = {
    "Higher-confidence controlled pilot": 1,
    "Higher-confidence cross-sell pilot": 2,
    "Limited-evidence pilot; repeat result but sparse": 3,
    "Affinity-led exploratory test; spending evidence is mixed": 4,
    "Relevant category, but do not discount automatically": 5,
    "Low priority until stronger evidence": 6,
}
category_decision["action_order"] = category_decision.suggested_action.map(action_order)

category_decision_display = category_decision[[
    "segment", "DEPARTMENT", "affinity_index", "affinity_label",
    "lift_pct_earlier", "lift_pct_later", "campaigns_earlier", "campaigns_later",
    "spending_evidence", "suggested_action", "action_order",
]].rename(columns={
    "segment": "Customer segment", "DEPARTMENT": "Coupon department",
    "affinity_index": "Pre-campaign affinity index",
    "affinity_label": "Normal purchasing pattern",
    "lift_pct_earlier": "Earlier matched spending lift",
    "lift_pct_later": "Later matched spending lift",
    "campaigns_earlier": "Earlier campaigns", "campaigns_later": "Later campaigns",
    "spending_evidence": "Coupon-linked spending evidence",
    "suggested_action": "Suggested use",
}).sort_values(
    ["Customer segment", "action_order", "Pre-campaign affinity index"],
    ascending=[True, True, False]
).drop(columns="action_order")

def joined_departments(frame):
    return ", ".join(frame.sort_values("affinity_index", ascending=False).DEPARTMENT)

higher_confidence = category_decision.loc[
    category_decision.suggested_action.str.startswith("Higher-confidence")
].groupby("segment").apply(joined_departments, include_groups=False).rename(
    "Higher-confidence controlled pilot"
)
exploratory = category_decision.loc[
    category_decision.suggested_action.isin([
        "Limited-evidence pilot; repeat result but sparse",
        "Affinity-led exploratory test; spending evidence is mixed",
    ])
].groupby("segment").apply(
    lambda frame: "; ".join(
        f"{row.DEPARTMENT} ({'sparse repeat' if row.suggested_action.startswith('Limited') else 'affinity-led'})"
        for row in frame.sort_values(["action_order", "affinity_index"], ascending=[True, False]).itertuples()
    ), include_groups=False
).rename("Exploratory tests")
priority_summary = pd.DataFrame(index=[NAMES[s] for s in SEG_ORDER]).join(
    higher_confidence
).join(exploratory).fillna("None").reset_index().rename(columns={"index": "Customer segment"})

method_table = pd.DataFrame({
    "Check": ["Pre-campaign category affinity", "Coupon-linked spending comparison", "Final interpretation"],
    "Simple meaning": [
        "Did the segment normally buy more of this department than average?",
        "Did recipients' department spending rise more than spending by similar non-recipients?",
        "Use both signals to choose controlled-pilot candidates; neither signal alone proves incremental profit.",
    ],
})

display(method_table.style.hide(axis="index").set_properties(
    subset=["Check", "Simple meaning"],
    **{"text-align": "left", "white-space": "normal", "max-width": "560px"}
).set_caption("How the two-check Type B/C category screen works"))
display(priority_summary.style.hide(axis="index").set_properties(
    subset=["Customer segment", "Higher-confidence controlled pilot", "Exploratory tests"],
    **{"text-align": "left", "white-space": "normal", "max-width": "760px"}
).set_caption("Shortlist for a controlled Type B/C bundle pilot"))
display(category_decision_display.style.hide(axis="index").format({
    "Pre-campaign affinity index": "{:.2f}",
    "Earlier matched spending lift": "{:+.1f}%",
    "Later matched spending lift": "{:+.1f}%",
    "Earlier campaigns": "{:,.0f}", "Later campaigns": "{:,.0f}",
}, na_rep="—").set_properties(
    subset=["Customer segment", "Coupon department", "Normal purchasing pattern",
            "Coupon-linked spending evidence", "Suggested use"],
    **{"text-align": "left", "white-space": "normal", "max-width": "320px"}
).set_caption(
    "Full segment–department evidence matrix: affinity establishes relevance; matched spending is confirmation"
))

### 4.3 Refine broad departments into commodities

Department results can be too broad: for example, **GROCERY** contains cheese, soft drinks, frozen pizza and many other distinct product types. This refinement asks whether commodity-level evidence produces more useful segment-specific bundles.

Only Type B/C coupon–campaign pairs whose eligible products all map to one commodity are used. To avoid recommendations driven by isolated observations, a commodity must appear in at least **3 B/C campaigns** and record at least **10 unambiguous redemptions overall**. The same earlier-to-later design is then repeated: learn the smallest commodity set capturing 80% of each segment's earlier redemptions, then evaluate it on later campaigns.

These remain **utilisation findings**, not causal incremental-spending estimates. A commodity can be recommended for a pilot because its coupons were used consistently, but a controlled test is still required to establish whether including it causes extra spending or profit.

In [ ]:
# Commodity-level refinement of the Type B/C bundle analysis.
# A coupon is assigned only when all eligible products map to one commodity.
bc_coupon_commodity_scope = bc_coupon_product_category.groupby(
    ["CAMPAIGN", "COUPON_UPC"]
).agg(
    commodity_count=("COMMODITY_DESC", "nunique"),
    DEPARTMENT=("DEPARTMENT", "first"),
    COMMODITY_DESC=("COMMODITY_DESC", "first"),
).reset_index()
bc_single_commodity_coupon_map = bc_coupon_commodity_scope.loc[
    bc_coupon_commodity_scope.commodity_count == 1,
    ["CAMPAIGN", "COUPON_UPC", "DEPARTMENT", "COMMODITY_DESC"],
].dropna(subset=["COMMODITY_DESC"])

bc_commodity_redemption_events = bc_raw_redemptions.merge(
    bc_single_commodity_coupon_map,
    on=["CAMPAIGN", "COUPON_UPC"], how="inner", validate="many_to_one",
)

commodity_coverage = bc_coupon_product_category.groupby(
    ["DEPARTMENT", "COMMODITY_DESC"]
).agg(
    campaigns=("CAMPAIGN", "nunique"),
    coupon_codes=("COUPON_UPC", "nunique"),
    eligible_products=("PRODUCT_ID", "nunique"),
)
commodity_redemptions = bc_commodity_redemption_events.groupby(
    ["DEPARTMENT", "COMMODITY_DESC"]
).size().rename("single_commodity_redemptions")
commodity_coverage = commodity_coverage.join(commodity_redemptions).fillna({
    "single_commodity_redemptions": 0
})
SUPPORTED_COMMODITIES = commodity_coverage.query(
    "campaigns >= 3 and single_commodity_redemptions >= 10"
).sort_values(
    ["single_commodity_redemptions", "campaigns"], ascending=False
).index

commodity_support_display = commodity_coverage.loc[SUPPORTED_COMMODITIES].reset_index().rename(columns={
    "DEPARTMENT": "Department",
    "COMMODITY_DESC": "Commodity",
    "campaigns": "B/C campaigns",
    "coupon_codes": "Coupon codes",
    "eligible_products": "Eligible products",
    "single_commodity_redemptions": "Observed redemptions",
})

# Every Type B/C recipient is exposed to each supported commodity represented
# by an unambiguous coupon in that campaign.
campaign_commodities = bc_single_commodity_coupon_map.merge(
    pd.DataFrame(SUPPORTED_COMMODITIES.tolist(), columns=["DEPARTMENT", "COMMODITY_DESC"]),
    on=["DEPARTMENT", "COMMODITY_DESC"], how="inner",
)[["CAMPAIGN", "DEPARTMENT", "COMMODITY_DESC"]].drop_duplicates()
commodity_exposure = table.loc[
    table.CAMPAIGN.isin(BC_CAMPAIGNS), ["household_key", "CAMPAIGN"]
].merge(campaign_commodities, on="CAMPAIGN", how="inner")
commodity_exposure["seg"] = commodity_exposure.household_key.map(H.seg)
commodity_exposure = commodity_exposure.dropna(subset=["seg"])
commodity_exposure["seg"] = commodity_exposure.seg.astype(int)

commodity_redemption = bc_commodity_redemption_events.loc[
    bc_commodity_redemption_events.set_index(["DEPARTMENT", "COMMODITY_DESC"]).index.isin(
        SUPPORTED_COMMODITIES
    )
].groupby(
    ["household_key", "CAMPAIGN", "DEPARTMENT", "COMMODITY_DESC"]
).size().rename("redemptions")
commodity_exposure = commodity_exposure.merge(
    commodity_redemption,
    left_on=["household_key", "CAMPAIGN", "DEPARTMENT", "COMMODITY_DESC"],
    right_index=True, how="left",
)
commodity_exposure["redeemed"] = commodity_exposure.redemptions.fillna(0).gt(0)
commodity_exposure = commodity_exposure.merge(
    desc[["CAMPAIGN", "START_DAY"]], on="CAMPAIGN", how="left", validate="many_to_one"
)
commodity_exposure["period"] = np.where(
    commodity_exposure.START_DAY < 500, "Earlier campaigns", "Later campaigns"
)

commodity_period_response = commodity_exposure.groupby(
    ["period", "seg", "DEPARTMENT", "COMMODITY_DESC"]
).agg(
    campaigns=("CAMPAIGN", "nunique"),
    commodity_exposures=("redeemed", "size"),
    redeemed_exposures=("redeemed", "sum"),
    redemption_rate=("redeemed", "mean"),
).reset_index()
commodity_period_response["segment"] = commodity_period_response.seg.map(NAMES)

earlier_commodity_response = commodity_period_response.loc[
    commodity_period_response.period == "Earlier campaigns"
].copy()

# Select the smallest commodity set capturing at least 80% of each segment's
# earlier observed redemptions, then freeze the rule for later validation.
commodity_selection_rows = []
for segment_id in SEG_ORDER:
    frame = earlier_commodity_response.loc[
        earlier_commodity_response.seg == segment_id
    ].sort_values(["redeemed_exposures", "redemption_rate"], ascending=False).copy()
    total_redemptions = frame.redeemed_exposures.sum()
    frame["cumulative_redemption_share"] = (
        frame.redeemed_exposures.cumsum() / total_redemptions if total_redemptions > 0 else 0
    )
    frame["selected_for_80pct_bundle"] = (
        frame.cumulative_redemption_share.shift(fill_value=0) < 0.80
    )
    commodity_selection_rows.append(frame)
commodity_bundle_selection = pd.concat(commodity_selection_rows, ignore_index=True)

selected_commodity_pairs = set(
    commodity_bundle_selection.loc[
        commodity_bundle_selection.selected_for_80pct_bundle,
        ["seg", "DEPARTMENT", "COMMODITY_DESC"],
    ].itertuples(index=False, name=None)
)
later_commodity_test = commodity_exposure.loc[
    commodity_exposure.period == "Later campaigns"
].copy()
later_commodity_test["selected_commodity"] = [
    (segment_id, department, commodity) in selected_commodity_pairs
    for segment_id, department, commodity in zip(
        later_commodity_test.seg,
        later_commodity_test.DEPARTMENT,
        later_commodity_test.COMMODITY_DESC,
    )
]

commodity_bundle_rows = []
for segment_id in SEG_ORDER:
    frame = later_commodity_test.loc[later_commodity_test.seg == segment_id]
    selected = frame.loc[frame.selected_commodity]
    selected_frame = commodity_bundle_selection.loc[
        (commodity_bundle_selection.seg == segment_id)
        & commodity_bundle_selection.selected_for_80pct_bundle
    ]
    selected_labels = (
        selected_frame.DEPARTMENT.str.title() + " — "
        + selected_frame.COMMODITY_DESC.str.title()
    ).tolist()
    all_redemptions = int(frame.redeemed.sum())
    selected_redemptions = int(selected.redeemed.sum())
    commodity_bundle_rows.append({
        "Customer segment": NAMES[segment_id],
        "Commodity bundle learned from earlier campaigns": "; ".join(selected_labels),
        "Earlier commodities observed": commodity_bundle_selection.loc[
            commodity_bundle_selection.seg == segment_id, "COMMODITY_DESC"
        ].nunique(),
        "Selected commodities": len(selected_labels),
        "Later commodity exposures retained": len(selected) / len(frame) if len(frame) else np.nan,
        "Later observed redemptions retained": (
            selected_redemptions / all_redemptions if all_redemptions else np.nan
        ),
        "All-commodity later redemption rate": frame.redeemed.mean() if len(frame) else np.nan,
        "Selected-commodity later redemption rate": selected.redeemed.mean() if len(selected) else np.nan,
        "Later redemptions": all_redemptions,
    })
commodity_bundle_holdout = pd.DataFrame(commodity_bundle_rows)
selected_commodity_all = later_commodity_test.loc[later_commodity_test.selected_commodity]
commodity_bundle_holdout = pd.concat([
    commodity_bundle_holdout,
    pd.DataFrame([{
        "Customer segment": "Overall — segment-specific rule",
        "Commodity bundle learned from earlier campaigns": "Varies by segment",
        "Earlier commodities observed": np.nan,
        "Selected commodities": np.nan,
        "Later commodity exposures retained": len(selected_commodity_all) / len(later_commodity_test),
        "Later observed redemptions retained": (
            selected_commodity_all.redeemed.sum() / later_commodity_test.redeemed.sum()
        ),
        "All-commodity later redemption rate": later_commodity_test.redeemed.mean(),
        "Selected-commodity later redemption rate": selected_commodity_all.redeemed.mean(),
        "Later redemptions": int(later_commodity_test.redeemed.sum()),
    }]),
], ignore_index=True)

overall_commodity_result = commodity_bundle_holdout.loc[
    commodity_bundle_holdout["Customer segment"] == "Overall — segment-specific rule"
].iloc[0]
commodity_decision_summary = pd.DataFrame({
    "Question": [
        "How much of the B/C coupon mapping is unambiguous at commodity level?",
        "How much evidence survives the minimum-support screen?",
        "Did the earlier segment-specific commodity bundles hold up later?",
        "How should commodity results be used?",
    ],
    "Finding": [
        (
            f"{len(bc_single_commodity_coupon_map) / len(bc_coupon_commodity_scope):.1%} of coupon–campaign pairs; "
            f"{len(bc_commodity_redemption_events) / len(bc_raw_redemptions):.1%} of B/C redemptions could be assigned to one commodity."
        ),
        f"{len(SUPPORTED_COMMODITIES)} commodities met the ≥3-campaign and ≥10-redemption screen.",
        (
            f"No. The rule retained {overall_commodity_result['Later commodity exposures retained']:.1%} of later exposures "
            f"but only {overall_commodity_result['Later observed redemptions retained']:.1%} of later redemptions; "
            f"the observed redemption rate was {overall_commodity_result['Selected-commodity later redemption rate']:.1%} "
            f"versus {overall_commodity_result['All-commodity later redemption rate']:.1%} across all supported commodities."
        ),
        (
            "Keep departments as the main bundle level. Use commodities only to design and test narrower pilots; "
            "do not present the historical commodity bundles as validated recommendations."
        ),
    ],
})

display(commodity_support_display.style.hide(axis="index").format({
    "B/C campaigns": "{:,.0f}", "Coupon codes": "{:,.0f}",
    "Eligible products": "{:,.0f}", "Observed redemptions": "{:,.0f}",
}).set_caption(
    "Commodity evidence retained after the ≥3-campaign and ≥10-redemption support screen"
))
display(commodity_bundle_holdout.style.hide(axis="index").format({
    "Earlier commodities observed": "{:,.0f}", "Selected commodities": "{:,.0f}",
    "Later commodity exposures retained": "{:.1%}",
    "Later observed redemptions retained": "{:.1%}",
    "All-commodity later redemption rate": "{:.1%}",
    "Selected-commodity later redemption rate": "{:.1%}",
    "Later redemptions": "{:,.0f}",
}, na_rep="—").set_properties(
    subset=["Commodity bundle learned from earlier campaigns"],
    **{"text-align": "left", "white-space": "normal", "max-width": "520px"}
).set_caption(
    "Commodity-level holdout test: segment-specific 80%-redemption bundles learned earlier"
))
display(commodity_decision_summary.style.hide(axis="index").set_properties(
    subset=["Question", "Finding"],
    **{"text-align": "left", "white-space": "normal", "max-width": "620px"}
).set_caption(
    "Decision from the commodity refinement: informative detail, but not a stable targeting rule"
))

## 5. Markov customer lifetime value

### CLV analysis — assess customer value and retention

**Purpose:** determine the long-term value and retention of each behavioural segment, providing secondary context for how much Kroger should invest in segments that respond positively to campaigns.

**Method:** shopping revenue is aggregated into 13-week quarters. Within each behavioural segment, households are classified quarterly as Top, Mid, Low or Inactive. The observed movement between these tiers is represented by a transition matrix and projected across 12 quarters, or three years.

Quarters are **anchored on the final week of data** (week 102), so every quarter used contains a full 13 weeks: quarter 7 = weeks 90–102, quarter 6 = weeks 77–89, and so on. Anchoring on week 1 instead leaves the final quarter with only 11 weeks (92–102), which understates recent spending, pushes households into lower tiers in the last quarter and lowers every segment's CLV by roughly 6–10%. Quarters 0–1 (weeks 1–24) are excluded to reduce panel ramp-up distortion. Expected revenue is converted to contribution using an **assumed 22% company-wide margin** and discounted at 10% annually. The result is a planning estimate of three-year contribution CLV, not observed accounting profit. CLV does not independently determine targeting; it indicates the scale of investment that a responsive segment may justify.

The same value-tier cut-offs are used for every segment, allowing their transition matrices to be compared directly. Four segment-specific matrices are followed by a summary of:

- **Retention:** probability that a Top household remains Top next quarter.
- **Upgrading:** probability that a Mid or Low household moves to a higher active tier.
- **Churn:** probability that an active household becomes Inactive.
- **Reactivation:** probability that an Inactive household becomes active again.

Very few Light shoppers ever reach the Top tier, so their Top-retention rate rests on a handful of transitions; the table reports how many transitions each rate is based on. **Active-to-Inactive churn is the more reliable loyalty comparison across all four segments.**

These transitions describe historical customer-value movement; they do not show that coupons caused the movement. A dedicated table below states every material CLV assumption and its implication before presenting the matrices and projected values. Section 5.1 then tests whether the CLV ranking survives different financial assumptions and sampling noise.


In [ ]:
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    hdr("6. CLV - quarterly value tiers + transition matrix (course M4)")
    # Quarters are anchored on the final week so every quarter used has a full 13 weeks
    # (q7 = weeks 90-102). Anchoring on week 1 left only 11 weeks (92-102) in the last quarter.
    last_week = int(shop.WEEK_NO.max())
    sq = shop.assign(q=(7 - (last_week - shop.WEEK_NO) // 13).clip(lower=0))
    Q = sq.groupby(["household_key", "q"]).SALES_VALUE.sum().unstack(fill_value=0).reindex(columns=range(8), fill_value=0)
    Q = Q.reindex(H.index)
    use = list(range(2, 8))                                         # drop quarters 0-1 = weeks 1-24 (panel ramp-up)
    q2 = Q[2][Q[2] > 0]; hi, mid = q2.quantile(.8), q2.quantile(.5)
    order = ["Top", "Mid", "Low", "Inactive"]
    T = pd.DataFrame({q: np.select([Q[q] >= hi, Q[q] >= mid, Q[q] > 0], order[:3], "Inactive") for q in use}, index=Q.index)
    tp = pd.concat([pd.DataFrame({"a": T[q].values, "b": T[q + 1].values}) for q in use[:-1]], ignore_index=True)
    M = pd.crosstab(tp.a, tp.b, normalize="index").reindex(index=order, columns=order).fillna(0)
    rev = pd.concat([pd.DataFrame({"tier": T[q].values, "rev": Q[q].values}) for q in use], ignore_index=True
                    ).groupby("tier").rev.mean().reindex(order)
    print(f"tier cut-offs per quarter: Top >= ${hi:.0f}, Mid >= ${mid:.0f}, Low > $0")
    print("transition matrix (rows = this quarter):\n" + (M * 100).round(1).to_string())
    dq = (1 + DISC_YR) ** 0.25 - 1
    def clv(start):
        v = np.array([1.0 if o == start else 0.0 for o in order]); tot = 0.0
        for k in range(1, HORIZON_Q + 1):
            v = v @ M.values; tot += (v * rev.values * MARGIN).sum() / (1 + dq) ** k
        return tot
    C = pd.Series({o: clv(o) for o in order})
    print("3-yr CLV by current tier (pooled matrix):", C.round(0).to_dict())

    # segment-specific matrices: each segment gets its own transitions and tier revenues (used in the team doc)
    def clv_segment(ids):
        tp_s = pd.concat([pd.DataFrame({"a": T.loc[ids, q].values, "b": T.loc[ids, q + 1].values}) for q in use[:-1]],
                         ignore_index=True)
        Ms = pd.crosstab(tp_s.a, tp_s.b, normalize="index").reindex(index=order, columns=order).fillna(0)
        rev_s = pd.concat([pd.DataFrame({"tier": T.loc[ids, q].values, "rev": Q.loc[ids, q].values}) for q in use],
                          ignore_index=True).groupby("tier").rev.mean().reindex(order).fillna(0)
        v = pd.Series(T.loc[ids, 7]).value_counts(normalize=True).reindex(order).fillna(0).values; tot = 0.0
        for k in range(1, HORIZON_Q + 1):
            v = v @ Ms.values; tot += (v * rev_s.values * MARGIN).sum() / (1 + dq) ** k
        return tot, Ms.loc["Top", "Top"]
    print("segment-specific 3-yr CLV:")
    for sg in SEG_ORDER:
        ids = H.index[H.seg == sg]; c, stay = clv_segment(ids)
        print(f"  {NAMES[sg]:20s} CLV/household ${c:,.0f}, segment total ${c * len(ids) / 1e6:.2f}M, Top->Top {stay:.0%}")

In [ ]:
tier_definition = pd.DataFrame({
    "Tier": ["Top", "Mid", "Low", "Inactive"],
    "Quarterly-spend definition": [
        f"At least ${hi:,.0f}", f"${mid:,.0f} to ${hi:,.0f}", "Above $0 below Mid", "$0"
    ],
})
pooled_clv = pd.DataFrame({
    "Mean quarterly revenue": rev,
    "Three-year revenue CLV": C / MARGIN,
    "Three-year contribution CLV (assumed 22% margin)": C,
})
segment_clv_rows = []
segment_transition_rows = []
segment_transition_matrices = {}
state_rank = {"Inactive": 0, "Low": 1, "Mid": 2, "Top": 3}
for segment_id in SEG_ORDER:
    segment_ids = H.index[H.seg == segment_id]
    segment_value, top_retention = clv_segment(segment_ids)
    segment_pairs = pd.concat([
        pd.DataFrame({
            "From": T.loc[segment_ids, quarter].values,
            "To": T.loc[segment_ids, quarter + 1].values,
        })
        for quarter in use[:-1]
    ], ignore_index=True)
    segment_matrix = pd.crosstab(
        segment_pairs["From"], segment_pairs["To"], normalize="index"
    ).reindex(index=order, columns=order).fillna(0)
    segment_transition_matrices[NAMES[segment_id]] = segment_matrix

    upgrade_origins = segment_pairs["From"].isin(["Mid", "Low"])
    active_origins = segment_pairs["From"] != "Inactive"
    inactive_origins = segment_pairs["From"] == "Inactive"
    upgraded = (
        segment_pairs.loc[upgrade_origins, "To"].map(state_rank).values
        > segment_pairs.loc[upgrade_origins, "From"].map(state_rank).values
    )
    segment_transition_rows.append({
        "Segment": NAMES[segment_id],
        "Observed transitions": len(segment_pairs),
        "Transitions from Top tier": int((segment_pairs["From"] == "Top").sum()),
        "Top retention": segment_matrix.loc["Top", "Top"],
        "Mid/Low upgrading": upgraded.mean() if upgrade_origins.any() else np.nan,
        "Active-to-Inactive churn": (
            segment_pairs.loc[active_origins, "To"].eq("Inactive").mean()
            if active_origins.any() else np.nan
        ),
        "Inactive-to-active reactivation": (
            segment_pairs.loc[inactive_origins, "To"].ne("Inactive").mean()
            if inactive_origins.any() else np.nan
        ),
    })
    segment_clv_rows.append({
        "Segment": NAMES[segment_id],
        "Households": len(segment_ids),
        "Revenue CLV per household": segment_value / MARGIN,
        "Contribution CLV per household (assumed 22% margin)": segment_value,
        "Segment revenue CLV": (segment_value / MARGIN) * len(segment_ids),
        "Segment contribution CLV (assumed 22% margin)": segment_value * len(segment_ids),
        "Top-to-Top transition": top_retention,
    })
segment_clv_table = pd.DataFrame(segment_clv_rows)
segment_transition_comparison = pd.DataFrame(segment_transition_rows).merge(
    segment_clv_table[[
        "Segment", "Revenue CLV per household",
        "Contribution CLV per household (assumed 22% margin)",
        "Segment contribution CLV (assumed 22% margin)",
    ]],
    on="Segment", how="left", validate="one_to_one",
)

clv_measure_definition = pd.DataFrame({
    "CLV measure": ["Three-year revenue CLV", "Three-year contribution CLV"],
    "Calculation": [
        "Discounted projected customer spending over 12 quarters",
        "Revenue CLV × assumed 22% company-wide gross margin",
    ],
    "Margin interpretation": [
        "Does not require a gross-margin assumption",
        "Planning estimate only; 22% is not a known segment- or category-specific margin",
    ],
})

clv_assumptions = pd.DataFrame({
    "Assumption": [
        "Projection horizon",
        "Quarter length",
        "Panel ramp-up treatment",
        "Value-tier thresholds",
        "Starting customer mix",
        "Transition stability",
        "Tier-revenue stability",
        "Annual discount rate",
        "Quarterly discount rate used",
        "Contribution margin",
        "Observed-customer scope",
        "Costs excluded from CLV",
    ],
    "Value or treatment": [
        f"{HORIZON_Q} quarters / {HORIZON_Q / 4:.0f} years",
        "13 weeks",
        "Quarters anchored on the final week (102), so each has 13 weeks; weeks 1–24 excluded; transitions use quarters 2–7",
        f"Common across segments: Top ≥ ${hi:,.0f}; Mid ≥ ${mid:,.0f}; Low > $0; Inactive = $0",
        "Each segment's observed state distribution in the latest quarter",
        "Each segment's historical transition matrix is assumed constant over the projection",
        "Historical mean revenue within each tier is assumed constant over the projection",
        f"{DISC_YR:.1%} per year",
        f"{dq:.2%} per quarter, calculated as (1 + annual rate)^(1/4) − 1",
        f"{MARGIN:.1%} company-wide gross margin, applied equally to every segment and tier",
        "Only recorded Kroger-panel spending is observed; spending elsewhere is unavailable",
        "Acquisition, servicing, coupon-campaign, processing and promotional-placement costs",
    ],
    "Implication": [
        "CLV is a three-year planning estimate rather than an indefinite lifetime value",
        "All state movements are measured at the same quarterly interval",
        "Avoids a short final quarter understating recent spend; reduces panel-entry distortion but shortens the transition history",
        "Makes segment matrices directly comparable; tiers are absolute panel-wide value states",
        "The projection begins from the segment's most recently observed mix of value states",
        "Structural changes in shopping behaviour would make the projection inaccurate",
        "Future spend within a tier may differ from its historical average",
        "Future value is discounted; the rate is an analyst assumption",
        "Ensures quarterly cash flows are discounted consistently with the annual assumption",
        "Contribution CLV is not known customer profit and ignores category-margin differences",
        "Panel exit or outside-retailer shopping may be misclassified as inactivity",
        "CLV measures projected revenue/contribution before these additional costs",
    ],
})

display(clv_measure_definition.style.hide(axis="index").set_properties(
    subset=["CLV measure", "Calculation", "Margin interpretation"],
    **{"text-align": "left", "white-space": "normal", "max-width": "520px"}
).set_caption("Revenue CLV and contribution CLV are different measures"))
display(clv_assumptions.style.hide(axis="index").set_properties(
    subset=["Assumption", "Value or treatment", "Implication"],
    **{"text-align": "left", "white-space": "normal", "max-width": "520px"}
).set_caption("CLV assumptions and their implications"))
display(tier_definition.style.hide(axis="index").set_caption("Quarterly value-tier definitions"))
display((M * 100).style.format("{:.1f}%").set_caption(
    "Pooled quarterly transition probabilities — rows are current tier, columns are next-quarter tier"
))
for segment_name in [NAMES[s] for s in SEG_ORDER]:
    display((segment_transition_matrices[segment_name] * 100).style.format("{:.1f}%").set_caption(
        f"{segment_name}: quarterly transition matrix — rows are current tier, columns are next-quarter tier"
    ))
display(pooled_clv.style.format({
    "Mean quarterly revenue": "${:,.2f}",
    "Three-year revenue CLV": "${:,.0f}",
    "Three-year contribution CLV (assumed 22% margin)": "${:,.0f}",
}).set_caption(
    "Pooled three-year revenue and contribution CLV — contribution uses an assumed 22% company-wide margin"
))
display(segment_clv_table.style.hide(axis="index").format({
    "Households": "{:,.0f}",
    "Revenue CLV per household": "${:,.0f}",
    "Contribution CLV per household (assumed 22% margin)": "${:,.0f}",
    "Segment revenue CLV": "${:,.0f}",
    "Segment contribution CLV (assumed 22% margin)": "${:,.0f}",
    "Top-to-Top transition": "{:.1%}",
}).set_caption(
    "Segment-specific three-year revenue and contribution CLV — 22% is an assumed common margin, not observed segment profitability"
))
display(segment_transition_comparison.style.hide(axis="index").format({
    "Observed transitions": "{:,.0f}",
    "Transitions from Top tier": "{:,.0f}",
    "Top retention": "{:.1%}",
    "Mid/Low upgrading": "{:.1%}",
    "Active-to-Inactive churn": "{:.1%}",
    "Inactive-to-active reactivation": "{:.1%}",
    "Revenue CLV per household": "${:,.0f}",
    "Contribution CLV per household (assumed 22% margin)": "${:,.0f}",
    "Segment contribution CLV (assumed 22% margin)": "${:,.0f}",
}).set_caption(
    "Why segment value differs: retention, upgrading, churn and reactivation compared with projected CLV"
))

### 5.1 Robustness of the CLV ranking

CLV depends on analyst assumptions (margin, discount rate, horizon) and on which households happen to be in each segment. Two checks test whether the **ranking of segments** — the only way CLV is used in the targeting decision — depends on those choices:

1. **Assumption sensitivity:** recompute segment CLV with the conservative and optimistic margins used in the financial scenarios (20.5% and 23.5%), discount rates of 5% and 15%, and horizons of two and five years.
2. **Sampling uncertainty:** resample households within each segment 500 times, re-estimate the transition matrix, tier revenues and starting mix each time, and report the 2.5th–97.5th percentile range of CLV per household.

If the ranking is unchanged across assumptions and the resampled ranges do not overlap, CLV is a stable input for sizing investment, even though the absolute dollar values are planning estimates.


In [ ]:
# CLV robustness: assumption sensitivity and household-resampling ranges.
def projected_clv(ids, margin=MARGIN, disc_yr=DISC_YR, horizon_q=HORIZON_Q, Qd=None, Td=None):
    """Segment-specific three-year contribution CLV per household, with adjustable assumptions."""
    Qd = Q if Qd is None else Qd
    Td = T if Td is None else Td
    pairs_s = pd.concat([pd.DataFrame({"a": Td.loc[ids, q].values, "b": Td.loc[ids, q + 1].values}) for q in use[:-1]],
                        ignore_index=True)
    matrix_s = pd.crosstab(pairs_s.a, pairs_s.b, normalize="index").reindex(index=order, columns=order).fillna(0)
    revenue_s = pd.concat([pd.DataFrame({"tier": Td.loc[ids, q].values, "rev": Qd.loc[ids, q].values}) for q in use],
                          ignore_index=True).groupby("tier").rev.mean().reindex(order).fillna(0)
    state = pd.Series(Td.loc[ids, use[-1]]).value_counts(normalize=True).reindex(order).fillna(0).values
    quarterly_rate = (1 + disc_yr) ** 0.25 - 1
    total = 0.0
    for k in range(1, horizon_q + 1):
        state = state @ matrix_s.values
        total += (state * revenue_s.values * margin).sum() / (1 + quarterly_rate) ** k
    return total

segment_names = [NAMES[s] for s in SEG_ORDER]
base_rank = None
sensitivity_rows = []
for case, kwargs in [
    ("Base: 22% margin, 10% discount, 3 years", {}),
    ("Conservative margin (20.5%)", {"margin": 0.205}),
    ("Optimistic margin (23.5%)", {"margin": 0.235}),
    ("Lower discount rate (5%)", {"disc_yr": 0.05}),
    ("Higher discount rate (15%)", {"disc_yr": 0.15}),
    ("Shorter horizon (2 years)", {"horizon_q": 8}),
    ("Longer horizon (5 years)", {"horizon_q": 20}),
]:
    values = {NAMES[s]: projected_clv(H.index[H.seg == s], **kwargs) for s in SEG_ORDER}
    rank = sorted(values, key=values.get, reverse=True)
    base_rank = rank if base_rank is None else base_rank
    sensitivity_rows.append({"Assumption case": case, **values, "Ranking unchanged?": "Yes" if rank == base_rank else "No"})
clv_sensitivity = pd.DataFrame(sensitivity_rows)

rng_clv = np.random.default_rng(0)
CLV_BOOTSTRAP = 500
interval_rows = []
for s in SEG_ORDER:
    ids = H.index[H.seg == s].to_numpy()
    draws = []
    for _ in range(CLV_BOOTSTRAP):
        sample = rng_clv.choice(ids, len(ids), replace=True)
        Qb, Tb = Q.loc[sample].reset_index(drop=True), T.loc[sample].reset_index(drop=True)
        draws.append(projected_clv(Qb.index, Qd=Qb, Td=Tb))
    interval_rows.append({
        "Segment": NAMES[s],
        "CLV per household": projected_clv(H.index[H.seg == s]),
        "95% lower": np.percentile(draws, 2.5),
        "95% upper": np.percentile(draws, 97.5),
    })
clv_intervals = pd.DataFrame(interval_rows)
ordered = clv_intervals.sort_values("CLV per household", ascending=False).reset_index(drop=True)
ranges_separate = all(ordered.loc[i, "95% lower"] > ordered.loc[i + 1, "95% upper"] for i in range(len(ordered) - 1))

display(clv_sensitivity.style.hide(axis="index").format({name: "${:,.0f}" for name in segment_names}).set_caption(
    "CLV per household under alternative assumptions — the segment ranking is the decision-relevant output"
))
display(clv_intervals.style.hide(axis="index").format({
    "CLV per household": "${:,.0f}", "95% lower": "${:,.0f}", "95% upper": "${:,.0f}",
}).set_caption(f"Household-resampling range for CLV per household ({CLV_BOOTSTRAP} resamples within each segment)"))
print("Ranking unchanged in every assumption case:", clv_sensitivity["Ranking unchanged?"].eq("Yes").all())
print("Resampled 95% ranges of adjacent segments do not overlap:", ranges_separate)


### 5.2 Presentation figure: segment value and loyalty

The figure combines the two CLV outputs used in the presentation: projected three-year contribution CLV per household (with its household-resampling range) and quarterly churn, the share of active households that become Inactive in the next quarter. Segment colours match the segmentation results.


In [ ]:
# Presentation figure: CLV per household (with resampling range) and quarterly churn by segment.
from matplotlib.ticker import FuncFormatter
SEGMENT_COLOURS = {
    "Big-basket regulars": "#003A8F", "Frequent top-up": "#2B8850",
    "Deal seekers": "#E26A0C", "Light shoppers": "#6F35AD",
}
fig_data = clv_intervals.merge(
    segment_transition_comparison[["Segment", "Active-to-Inactive churn"]], on="Segment", validate="one_to_one"
).merge(segment_clv_table[["Segment", "Segment contribution CLV (assumed 22% margin)"]], on="Segment", validate="one_to_one")
fig_data = fig_data.sort_values("CLV per household", ascending=True).reset_index(drop=True)
colours = fig_data["Segment"].map(SEGMENT_COLOURS)

fig, (ax_clv, ax_churn) = plt.subplots(1, 2, figsize=(12, 4.6), dpi=140,
                                       gridspec_kw={"width_ratios": [2.2, 1]}, sharey=True)
ax_clv.barh(fig_data["Segment"], fig_data["CLV per household"], color=colours, height=0.62)
ax_clv.errorbar(
    fig_data["CLV per household"], fig_data.index,
    xerr=[fig_data["CLV per household"] - fig_data["95% lower"], fig_data["95% upper"] - fig_data["CLV per household"]],
    fmt="none", ecolor="#374151", elinewidth=1.2, capsize=3,
)
for y, row in fig_data.iterrows():
    # "\$" stops matplotlib reading a pair of dollar signs as maths notation
    ax_clv.text(row["95% upper"] + 40, y,
                f"\\${row['CLV per household']:,.0f}   (segment total \\${row['Segment contribution CLV (assumed 22% margin)'] / 1e6:.2f}M)",
                va="center", fontsize=10)
ax_clv.set_xlim(0, fig_data["95% upper"].max() * 1.75)
ax_clv.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"\\${x:,.0f}"))
ax_clv.set_xlabel("Three-year contribution CLV per household (assumed 22% margin)")
ax_clv.set_title("A. Projected value per household", weight="bold", loc="left")
ax_churn.barh(fig_data["Segment"], fig_data["Active-to-Inactive churn"], color=colours, height=0.62)
for y, value in enumerate(fig_data["Active-to-Inactive churn"]):
    ax_churn.text(value + 0.002, y, f"{value:.1%}", va="center", fontsize=10)
ax_churn.set_xlim(0, fig_data["Active-to-Inactive churn"].max() * 1.45)
ax_churn.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
ax_churn.set_xlabel("Share of active households becoming Inactive next quarter")
ax_churn.set_title("B. Quarterly churn", weight="bold", loc="left")
for ax in (ax_clv, ax_churn):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", visible=False)
fig.suptitle("Loyal segments are worth more: lower churn, higher three-year value", fontsize=14, weight="bold", y=1.02)
fig.text(0.5, -0.04,
         "Error bars: 2.5th–97.5th percentile of CLV across 500 household resamples. "
         "CLV is a planning estimate; 22% is an assumed company-wide margin.",
         ha="center", fontsize=9, color="#4B5563")
plt.tight_layout()
plt.show()


## 6. Campaign lift with matched difference-in-differences

### Assess whether campaigns generate additional incremental spending

This is to determine which behavioural segments appear to spend more because they received a campaign—not merely which segments redeemed coupons.

Within each segment, compare recipients' change in spending from before to during the campaign with the change among similar non-recipients. Positive matched lift means recipients' spending increased more than the comparison group.

For each campaign, recipient spending changes from an eight-week pre-period to the campaign period. Three control definitions are compared: households never targeted,households matched on eight-week pre-spend, and
households matched jointly on 26-week and eight-week pre-spend.

We decided to use the third option as the base case. Segment uncertainty is obtained by resampling campaigns. This makes the interval reflect campaign-to-campaign variation, however, it does not solve every causal concern whereby recipients may still differ on unobserved factors, parallel trends are assumed within matching bins, and overlapping campaigns can complicate attribution.

In [ ]:
#campaign lift calcualtion method
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    hdr("7. CAMPAIGN LIFT - difference-in-differences, 3 comparison groups")
    daily = shop.groupby(["household_key", "DAY"]).SALES_VALUE.sum().reset_index()
    never = set(H.index) - set(hc.household_key); out = []; seg_rows = []
    for c, g in hc.groupby("CAMPAIGN"):
        st, en, typ = g.START_DAY.iloc[0], g.END_DAY.iloc[0], g.campaign_type.iloc[0]
        win = lambda a, z: daily[(daily.DAY >= a) & (daily.DAY <= z)].groupby("household_key").SALES_VALUE.sum().reindex(H.index).fillna(0)
        pre8, pre26, dur = win(st - 56, st - 1) / 56, win(st - 182, st - 1) / 182, win(st, en) / (en - st + 1)
        d = pd.DataFrame({"pre": pre8, "long": pre26, "chg": dur - pre8}); d["treat"] = d.index.isin(g.household_key)
        naive = d[d.treat].chg.mean() - d[d.index.isin(never)].chg.mean()
        def matched(bins):
            mm = d.groupby([bins, "treat"]).chg.mean().unstack().dropna(); n = d[d.treat].groupby(bins).size().reindex(mm.index).fillna(0)
            return ((mm[True] - mm[False]) * n).sum() / n.sum()
        d["b1"] = pd.qcut(d.pre.rank(method="first"), 20, labels=False)
        d["b2"] = pd.qcut(d.long.rank(method="first"), 10, labels=False) * 10 + pd.qcut(d.pre.rank(method="first"), 5, labels=False)
        out.append(dict(c=c, type=typ, n=int(d.treat.sum()), days=en - st + 1, base=d[d.treat].pre.mean(),
                        naive=naive, m8=matched("b1"), m26=matched("b2")))
        # lift by segment: each recipient vs non-recipients in the same 26+8-week spend
        d["seg"] = H.seg; ctl = d[~d.treat].groupby("b2").chg.mean()
        trt = d[d.treat].assign(ctl=lambda x: x.b2.map(ctl)).dropna(subset=["ctl"])
        for sg, x in trt.groupby("seg"):
            seg_rows.append(dict(c=c, seg=sg, n=len(x), did=(x.chg - x.ctl).mean(), base=x.pre.mean()))
    o = pd.DataFrame(out); wavg = lambda d, col: (d[col] * d.n).sum() / d.n.sum()
    for col, lab in [("naive", "vs never-targeted"), ("m8", "matched 8-wk pre-spend"), ("m26", "matched 26+8-wk pre-spend")]:
        print(f"{lab:28s} lift ${wavg(o, col):+.2f}/day ({100 * wavg(o, col) / wavg(o, 'base'):+.1f}%), "
              f"positive in {(o[col] > 0).sum()}/30 campaigns")
    for typ, d in o.groupby("type"):
        print(f"  {typ}: base-case extra spend per household per campaign ${wavg(d, 'm26') * wavg(d, 'days'):.1f}, "
              f"redemptions per recipient {len(cr[cr.campaign_type == typ]) / len(hc[hc.campaign_type == typ]):.2f}")
    so = pd.DataFrame(seg_rows); pairs["seg"] = pairs.household_key.map(H.seg)
    rng = np.random.default_rng(0); camps = so.c.unique()
    print("by segment (tight match): redemption rate | lift | 95% bootstrap range over campaigns | campaigns positive")
    for sg in SEG_ORDER:
        x = so[so.seg == sg].set_index("c"); est = []
        for _ in range(2000):
            xx = x.reindex(rng.choice(camps, len(camps), replace=True)).dropna()
            est.append(100 * (xx.did * xx.n).sum() / (xx.base * xx.n).sum())
        print(f"  {NAMES[sg]:20s} redeem {pairs[pairs.seg == sg].y.mean():.1%} | lift {100 * (x.did * x.n).sum() / (x.base * x.n).sum():+.1f}% | "
              f"[{np.percentile(est, 2.5):+.1f}%, {np.percentile(est, 97.5):+.1f}%] | {(x.did > 0).sum()}/{len(x)}")
    x = cr.merge(cm, on=["COUPON_UPC", "CAMPAIGN"]).merge(
        t[t.COUPON_DISC < 0][["household_key", "DAY", "PRODUCT_ID", "COUPON_DISC"]], on=["household_key", "DAY", "PRODUCT_ID"]
    ).drop_duplicates(["household_key", "DAY", "COUPON_UPC"])
    print(f"coupon value per matched redemption: median ${-x.COUPON_DISC.median():.2f}, mean ${-x.COUPON_DISC.mean():.2f} (n={len(x)})")

In [ ]:
#table to reference for slide 
campaign_lift_table = o.rename(columns={
    "c": "campaign",
    "n": "recipients",
    "base": "pre_spend_per_day",
    "naive": "lift_vs_never",
    "m8": "lift_matched_8w",
    "m26": "lift_matched_26w_8w",
})
lift_method_summary = pd.DataFrame({
    "Comparison group": ["Never targeted", "Matched on 8-week spend", "Matched on 26- and 8-week spend"],
    "Daily lift": [wavg(o, "naive"), wavg(o, "m8"), wavg(o, "m26")],
    "Lift relative to recipient baseline": [
        wavg(o, "naive") / wavg(o, "base"),
        wavg(o, "m8") / wavg(o, "base"),
        wavg(o, "m26") / wavg(o, "base"),
    ],
    "Campaigns with positive lift": [
        int((o.naive > 0).sum()), int((o.m8 > 0).sum()), int((o.m26 > 0).sum())
    ],
})

segment_rows = []
segment_rng = np.random.default_rng(4211)
for segment_id in SEG_ORDER:
    z = so[so.seg == segment_id].set_index("c")
    point_lift = (z.did * z.n).sum() / (z.base * z.n).sum()
    bootstrap_lift = []
    for _ in range(1000):
        sample = z.reindex(segment_rng.choice(camps, len(camps), replace=True)).dropna()
        bootstrap_lift.append((sample.did * sample.n).sum() / (sample.base * sample.n).sum())
    segment_rows.append({
        "Segment": NAMES[segment_id],
        "Redemption rate": pairs.loc[pairs.seg == segment_id, "y"].mean(),
        "Matched spending lift": point_lift,
        "95% lower": np.percentile(bootstrap_lift, 2.5),
        "95% upper": np.percentile(bootstrap_lift, 97.5),
        "Positive campaigns": int((z.did > 0).sum()),
        "Campaigns observed": len(z),
    })
segment_lift_summary = pd.DataFrame(segment_rows)

#overall table to reference from
display(lift_method_summary.style.hide(axis="index").format({
    "Daily lift": "${:+.2f}",
    "Lift relative to recipient baseline": "{:+.1%}",
    "Campaigns with positive lift": "{:.0f} of 30",
}).set_caption("Campaign lift under alternative comparison groups"))
display(segment_lift_summary.style.hide(axis="index").format({
    "Redemption rate": "{:.1%}",
    "Matched spending lift": "{:+.1%}",
    "95% lower": "{:+.1%}",
    "95% upper": "{:+.1%}",
    "Positive campaigns": "{:,.0f}",
    "Campaigns observed": "{:,.0f}",
}).set_caption("Matched campaign lift by customer segment: all 30 campaigns "
             "(reference only; decisions use Type B/C campaigns, Section 6.0a)"))
display(campaign_lift_table.style.hide(axis="index").format({
    "recipients": "{:,.0f}", "days": "{:,.0f}",
    "pre_spend_per_day": "${:.2f}", "lift_vs_never": "${:+.2f}",
    "lift_matched_8w": "${:+.2f}", "lift_matched_26w_8w": "${:+.2f}",
}).set_caption("Campaign-level lift estimates"))

### All campaigns versus Type B/C campaigns only

The segment lift above pools all 30 campaigns, including the 5 personalised Type A campaigns. Because the targeting decision concerns Type B/C campaigns, the same matched estimate is recomputed using only the 25 Type B/C campaigns. The method, comparison cells, weighting and bootstrap (1,000 campaign resamples) are identical; only the set of campaigns changes.

The decision rule is applied to both versions: 
- the whole 95% range above zero → *Prioritise*; 
- the whole range below zero → *Deprioritise*; 
- otherwise → *Conditional Pilots*. 

A segment whose recommendation differs between the two versions is less robust and should be presented with that caveat.

The decision table, ROI, and priority matrix use the Type B/C-only estimates from this check, because Type A is kept unchanged and only Type B/C distribution is being redesigned. The pooled all-30 estimate above is kept for reference.


In [ ]:

BC_LIFT_CAMPAIGNS = set(o.loc[o.type.isin(["TypeB", "TypeC"]), "c"])
ALL_LIFT_CAMPAIGNS = set(o.c)

#segment camapaign ligft diagram / table
def segment_lift_by_campaign_set(campaign_ids, draws=1000, seed=4211):
    """Matched segment lift and a campaign-level bootstrap range for one set of campaigns."""
    campaign_ids = np.array(sorted(campaign_ids))
    rng = np.random.default_rng(seed)
    rows = []
    for segment_id in SEG_ORDER:
        z = so[(so.seg == segment_id) & so.c.isin(campaign_ids)].set_index("c")
        boot = []
        for _ in range(draws):
            sample = z.reindex(rng.choice(campaign_ids, len(campaign_ids), replace=True)).dropna()
            boot.append((sample.did * sample.n).sum() / (sample.base * sample.n).sum())
        rows.append({
            "Segment": NAMES[segment_id],
            "Lift": (z.did * z.n).sum() / (z.base * z.n).sum(),
            "95% lower": np.percentile(boot, 2.5),
            "95% upper": np.percentile(boot, 97.5),
            "Positive campaigns": f"{int((z.did > 0).sum())} of {len(z)}",
            "Recipient pairs": int(z.n.sum()),
        })
    return pd.DataFrame(rows)


def rule(row):
    if row["95% lower"] > 0:
        return "Prioritise"
    if row["95% upper"] < 0:
        return "Deprioritise"
    return "Controlled tests"


lift_all = segment_lift_by_campaign_set(ALL_LIFT_CAMPAIGNS)
lift_bc = segment_lift_by_campaign_set(BC_LIFT_CAMPAIGNS)
for frame in (lift_all, lift_bc):
    frame["Decision rule"] = frame.apply(rule, axis=1)

#all segments
assert np.allclose(lift_all[["Lift", "95% lower", "95% upper"]].to_numpy(),
                   segment_lift_summary[["Matched spending lift", "95% lower", "95% upper"]].to_numpy())

lift_robustness = lift_all.merge(lift_bc, on="Segment", suffixes=(" (all 30)", " (B/C only)"))
lift_robustness["Decision changes?"] = np.where(
    lift_robustness["Decision rule (all 30)"] == lift_robustness["Decision rule (B/C only)"], "No", "Yes"
)

#robustness check
fmt_range = lambda lo, hi: f"[{lo:+.1%}, {hi:+.1%}]"
lift_robustness_display = pd.DataFrame({
    "Segment": lift_robustness["Segment"],
    "All 30: lift": lift_robustness["Lift (all 30)"],
    "All 30: 95% range": [fmt_range(a, b) for a, b in zip(lift_robustness["95% lower (all 30)"], lift_robustness["95% upper (all 30)"])],
    "All 30: positive": lift_robustness["Positive campaigns (all 30)"],
    "All 30: rule": lift_robustness["Decision rule (all 30)"],
    "B/C only: lift": lift_robustness["Lift (B/C only)"],
    "B/C only: 95% range": [fmt_range(a, b) for a, b in zip(lift_robustness["95% lower (B/C only)"], lift_robustness["95% upper (B/C only)"])],
    "B/C only: positive": lift_robustness["Positive campaigns (B/C only)"],
    "B/C only: rule": lift_robustness["Decision rule (B/C only)"],
    "Decision changes?": lift_robustness["Decision changes?"],
})
display(lift_robustness_display.style.hide(axis="index").format({
    "All 30: lift": "{:+.1%}", "B/C only: lift": "{:+.1%}",
}).set_caption(
    f" All {len(ALL_LIFT_CAMPAIGNS)} campaigns vs {len(BC_LIFT_CAMPAIGNS)} Type B/C campaigns only"
))

# Side-by-side interval plot.
fig, ax = plt.subplots(figsize=(9, 4.2), dpi=140)
labels = lift_robustness["Segment"].tolist()[::-1]
y = np.arange(len(labels))
for offset, suffix, colour, name in [(0.15, " (all 30)", "#24478F", f"All {len(ALL_LIFT_CAMPAIGNS)} campaigns"),
                                     (-0.15, " (B/C only)", "#F59E0B", f"Type B/C only ({len(BC_LIFT_CAMPAIGNS)})")]:
    d = lift_robustness.iloc[::-1]
    ax.errorbar(d["Lift" + suffix], y + offset,
                xerr=[d["Lift" + suffix] - d["95% lower" + suffix], d["95% upper" + suffix] - d["Lift" + suffix]],
                fmt="o", color=colour, ecolor=colour, elinewidth=2, capsize=3, markersize=6, label=name)
ax.axvline(0, color="#6B7280", linestyle="--", linewidth=1)
ax.set_yticks(y, labels)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Matched campaign spending lift (95% campaign-bootstrap range)")
ax.set_title("Segment lift: all campaigns vs Type B/C campaigns only", weight="bold")
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

changed = lift_robustness.loc[lift_robustness["Decision changes?"] == "Yes", "Segment"].tolist()
print("Segments whose rule-based decision changes when only Type B/C campaigns are used:",
      ", ".join(changed) if changed else "none")


# B/C lift
bc_so = so[so.c.isin(BC_LIFT_CAMPAIGNS)]
bc_pairs = pairs[pairs.CAMPAIGN.isin(BC_LIFT_CAMPAIGNS)]
segment_lift_summary_bc = pd.DataFrame({
    "Segment": [NAMES[s] for s in SEG_ORDER],
    "Redemption rate": [bc_pairs.loc[bc_pairs.seg == s, "y"].mean() for s in SEG_ORDER],
    "Matched spending lift": lift_bc["Lift"].to_numpy(),
    "95% lower": lift_bc["95% lower"].to_numpy(),
    "95% upper": lift_bc["95% upper"].to_numpy(),
    "Positive campaigns": [int((bc_so[bc_so.seg == s].did > 0).sum()) for s in SEG_ORDER],
    "Campaigns observed": [int((bc_so.seg == s).sum()) for s in SEG_ORDER],
})
assert (segment_lift_summary_bc["Segment"].to_numpy() == lift_bc["Segment"].to_numpy()).all()
display(segment_lift_summary_bc.style.hide(axis="index").format({
    "Redemption rate": "{:.1%}",
    "Matched spending lift": "{:+.1%}",
    "95% lower": "{:+.1%}",
    "95% upper": "{:+.1%}",
    "Positive campaigns": "{:,.0f}",
    "Campaigns observed": "{:,.0f}",
}).set_caption(f"Matched campaign lift by customer segment: {len(BC_LIFT_CAMPAIGNS)} Type B/C campaigns)"))


### 6.1 Decision use: combining campaign lift with CLV

Prioritise segments with reliable positive campaign lift, using CLV as secondary context for the scale of investment. The category analysis could not reliably prescribe exact segment-specific coupon bundles: department findings were too broad, while commodity findings were sparse and unstable in later campaigns. The stronger decision is therefore which behavioural segments should receive Type B/C campaigns more selectively.

All lift and redemption figures in this section are for the 25 Type B/C campaigns, because campaign lift for TypeA+B+C have already been included in the code above. A segment whose central lift is positive but whose range includes zero is recommended for a controlled pilot rather than a full roll-out.

In [ ]:
type_bc_targeting_decision = segment_clv_table.merge(
    segment_lift_summary_bc, on="Segment", how="inner", validate="one_to_one"
)
type_bc_targeting_decision["Statistically positive response"] = (
    type_bc_targeting_decision["95% lower"] > 0
)

#criteria
def audience_action(row):
    if row["95% lower"] > 0:
        return "Prioritise for Type B/C campaigns"
    if row["Matched spending lift"] < 0 and row["95% upper"] < 0:
        return "Deprioritise broad Type B/C distribution"
    if row["Matched spending lift"] > 0:
        return "Pilot with a controlled holdout before scaling"
    return "Limit distribution; use only controlled tests"

def audience_reason(row):
    clv = row["Contribution CLV per household (assumed 22% margin)"]
    if row["95% lower"] > 0:
        return (
            "Matched spending lift is positive even at the lower bound of the range; "
            f"three-year contribution CLV is ${clv:,.0f} per household under the assumed margin."
        )
    if row["Matched spending lift"] < 0 and row["95% upper"] < 0:
        return (
            "The matched lift estimate is below zero across the range, so there is no evidence that broad "
            f"distribution raises total spending; contribution CLV is ${clv:,.0f} per household."
        )
    if row["Matched spending lift"] > 0:
        return (
            "Central lift is positive but the range includes zero; with contribution CLV of "
            f"${clv:,.0f} per household, a controlled holdout test can confirm the response before scaling."
        )
    return (
        "Coupon redemption does not translate into a reliable increase in total spending; "
        "the range includes no effect."
    )

type_bc_targeting_decision["Recommended Type B/C treatment"] = type_bc_targeting_decision.apply(
    audience_action, axis=1
)
type_bc_targeting_decision["Reason"] = type_bc_targeting_decision.apply(audience_reason, axis=1)

audience_decision_display = type_bc_targeting_decision[[
    "Segment", "Households", "Revenue CLV per household",
    "Contribution CLV per household (assumed 22% margin)",
    "Segment contribution CLV (assumed 22% margin)",
    "Redemption rate", "Matched spending lift", "95% lower", "95% upper",
    "Recommended Type B/C treatment", "Reason",
]].copy()

#to include for explanation
decision_logic = pd.DataFrame({
    "Decision input": [
        "Matched total-spending response", "Three-year CLV", "Redemption rate",
        "Department/commodity analysis",
    ],
    "Role in the decision": [
        "Primary: determines whether the campaign appears to change spending.",
        "Secondary: determines how much investment a responsive segment may justify.",
        "Diagnostic only: coupon use does not necessarily mean additional spending.",
        "Exploratory: informs future bundle pilots, but is not stable enough to prescribe exact bundles.",
    ],
})

#table
display(decision_logic.style.hide(axis="index").set_properties(
    subset=["Decision input", "Role in the decision"],
    **{"text-align": "left", "white-space": "normal", "max-width": "620px"}
).set_caption("How the revised Type B/C targeting decision is made"))
display(audience_decision_display.style.hide(axis="index").format({
    "Households": "{:,.0f}", "Revenue CLV per household": "${:,.0f}",
    "Contribution CLV per household (assumed 22% margin)": "${:,.0f}",
    "Segment contribution CLV (assumed 22% margin)": "${:,.0f}",
    "Redemption rate": "{:.1%}",
    "Matched spending lift": "{:+.1%}", "95% lower": "{:+.1%}", "95% upper": "{:+.1%}",
}).set_properties(
    subset=["Segment", "Recommended Type B/C treatment", "Reason"],
    **{"text-align": "left", "white-space": "normal", "max-width": "440px"}
))

## 7. Do displays and mailers reinforce coupon campaigns?

### Supplementary measure — display and mailer promotional support

**Purpose:** Identify which forms of promotional support are most associated with stronger sales revenue of coupon-eligible products. Compare 3 promotional types: display-only, mailer-only, and combined (display and mailer support).

Coupon campaign information is linked to products and their active campaign weeks to identify coupon-eligible product–store–week observations. We then compare the sales revenue across the three promotional types.

An overall descriptive comparison of ave sales revenue per product-store-week is done first, to see how sales differ across the 3 promotional types.

Next, a 2 way fixed effect regression is run. Accounting for product-store fixed effect and week-fixed effect. The reference category used is mailer-only.

Wald-test is also done to estimate if the difference between the combined and display-only are statistically significant. 

Note that this analysis uses 5% of the original data (because of laptop capacity constraints), and only on coupon-eligible observations. Also note that the results are more correlational rather than casual, and that the dependent revenue is sales revenue and not profits (as such cost still needs to be accounted for eventually). 


In [ ]:
# descriptive analytics and panel reg 
import duckdb
import pandas as pd
from linearmodels.panel import PanelOLS

con = duckdb.connect()

#im just taking the coupon-eligible obs since that's the focus. -> take just 5% cause my laptop cannot load the full one 
df = con.execute("""
    SELECT
        HASH(PRODUCT_ID, STORE_ID) AS entity_id,
        WEEK_NO,
        promotion_type,
        sales_revenue,
        quantity_sold,
        display_only,
        combined
    FROM read_parquet('processed_data/regression_panel.parquet')
    WHERE coupon_eligible = 1
      AND HASH(PRODUCT_ID, STORE_ID) % 20 = 0
      AND promotion_type IN ('combined', 'display_only', 'mailer_only')
""").df()

#print(df)
#checking the details 
print("Number of observations:", len(df))
print("Number of product-store groups:", df["entity_id"].nunique())
print("Number of weeks:", df["WEEK_NO"].nunique())


#basic descriptive comparison first, later then do 2 fe reg
descriptive_table = df.groupby("promotion_type").agg(
    obs=("sales_revenue", "count"),
    total_qty=("quantity_sold", "sum"),
    total_sales=("sales_revenue", "sum"),
    avg_qty_per_obs=("quantity_sold", "mean"),
    avg_sales_per_obs=("sales_revenue", "mean")
).reindex([
    "combined",
    "display_only",
    "mailer_only"
]).reset_index()

print("descriptive stats compare")
display(descriptive_table.round(4))

# panel data for reg 
#Convert entity ID to string for the panel index.
df["entity_id"] = df["entity_id"].astype(str)
df = df.set_index(["entity_id", "WEEK_NO"]).sort_index() #for the store-pdt fixed (entity) effect and time fixed effect 

#Dependent variable: weekly sales revenue
y = df["sales_revenue"]

#Explanatory variables: (use the mailer as a ref category )
X = df[["display_only", "combined"]]


# two way fixed effect reg run 
model = PanelOLS(y,X, entity_effects=True,time_effects=True,drop_absorbed=True) #remove the perfect colinearty one 
results = model.fit(cov_type="clustered",cluster_entity=True,low_memory=True)

print("two-way fixed effect reg")
print(results.summary)

#doing further comparison of combined vs display-only 
combined_vs_display_test = results.wald_test(formula="combined = display_only")
#estimated coefficient difference and the p value for it 
difference = results.params["combined"] - results.params["display_only"]
p_value = combined_vs_display_test.pval
print("Coefficient difference:", difference.round(4))
print("p-value:", p_value.round(4))

#close duckdb  
con.close()


## 8. Recommendations

### 8.1 Assess financial viability of segment-selective Type B/C targeting

**Purpose:** determine whether targeting only campaign-responsive segments is financially preferable to distributing Type B/C campaigns broadly.

Only recipient–campaign pairs that have both a segment and an available matched comparison receive an uplift estimate. A household receiving multiple campaigns counts once for each campaign received. The analysis compares:

1. **Broad matched-recipient distribution:** retain all recorded recipient–campaign pairs with a valid segment and matched comparison. This is not every household in the dataset.
2. **Responsive-segment distribution:** retain only Big-basket regulars (reliably positive lift) and Frequent top-up shoppers (positive central lift, run as a controlled pilot).

**Both strategies use only the 25 Type B/C campaigns.** Type A is excluded because the proposal keeps it unchanged and its personalised coupon selections are not recorded, so segment uplifts are the Type B/C estimates. A per-segment table also shows each segment's incremental contribution after its own distribution cost.

The central uplift uses the matched campaign-lift estimate. Conservative and optimistic uplifts use the lower and upper campaign-resampling estimates. Financial scenarios separately vary the assumed gross margin, per-contact distribution cost and fixed creative cost. These scenario inputs are not observed Kroger campaign costs.

Incremental sales use recorded `SALES_VALUE`, which is already measured after recorded discounts; coupon face value is therefore not subtracted again. Coupon-processing, unobserved manufacturer-funding arrangements and display/mailer placement costs are unavailable and excluded. Section 7 reports placement associations separately and does not treat promotional support as free.

The section reports contacts avoided, campaign-cost savings, incremental contribution, estimated net profit, ROI and break-even conditions. These are retrospective planning scenarios—not observed campaign profit or proof that future targeting will achieve the same response.

In [ ]:
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    hdr("9. ROI - broad matched-recipient distribution vs responsive-segment distribution (Type B/C only)")
    RESPONSIVE = ["Big-basket regulars", "Frequent top-up"]
    BROAD_STRATEGY = "Broad matched-recipient distribution"
    TARGETED_STRATEGY = "Responsive-segment distribution"

    # Type B/C campaigns only: Type A is kept unchanged, so it is not part of the targeting decision.
    ROI_CAMPAIGNS = BC_LIFT_CAMPAIGNS
    hc_roi = hc[hc.CAMPAIGN.isin(ROI_CAMPAIGNS)]
    so_roi = so[so.c.isin(ROI_CAMPAIGNS)]
    o_roi = o[o.c.isin(ROI_CAMPAIGNS)]
    assert set(hc_roi.campaign_type) == {"TypeB", "TypeC"}

    # Use the same matched recipients as Section 6.
    eligible = []
    for c, g in hc_roi.groupby("CAMPAIGN"):
        st, en = int(g.START_DAY.iloc[0]), int(g.END_DAY.iloc[0])
        win = lambda a, z: daily[(daily.DAY >= a) & (daily.DAY <= z)].groupby("household_key").SALES_VALUE.sum().reindex(H.index).fillna(0)
        pre8, pre26 = win(st - 56, st - 1) / 56, win(st - 182, st - 1) / 182
        q = pd.DataFrame({"pre": pre8, "long": pre26})
        q["treat"] = q.index.isin(g.household_key)
        q["bin"] = pd.qcut(q.long.rank(method="first"), 10, labels=False) * 10 + pd.qcut(q.pre.rank(method="first"), 5, labels=False)
        controls = set(q.loc[~q.treat, "bin"])
        q = q[q.treat & q.bin.isin(controls)].copy()
        q["c"] = c; q["seg"] = H.seg; q["days"] = en - st + 1
        eligible.append(q.rename_axis("household_key").reset_index())
    e = pd.concat(eligible, ignore_index=True)
    assert not e.duplicated(["household_key", "c"]).any()
    assert e.groupby(["c", "seg"]).size().sort_index().equals(so_roi.set_index(["c", "seg"]).n.sort_index())
    r = cr.drop_duplicates(["household_key", "CAMPAIGN", "DAY", "COUPON_UPC"]).groupby(["household_key", "CAMPAIGN"]).size().rename("redemptions")
    e = e.merge(r, left_on=["household_key", "c"], right_index=True, how="left", validate="one_to_one")
    e["redemptions"] = e.redemptions.fillna(0)
    e["segment"] = e.seg.map(NAMES)
    print(f"recorded Type B/C recipient pairs {len(hc_roi)}, eligible matched pairs {len(e)}, unique eligible households {e.household_key.nunique()}")
    print("Excluded pairs have no segment or no matched comparison; no lift is assigned to them.")

    # Resample campaigns to get lower and upper uplift estimates.
    rr = np.random.default_rng(4211); lift_rows = []
    for sg in SEG_ORDER:
        z = so_roi[so_roi.seg == sg].merge(o_roi[["c", "days"]], on="c", validate="many_to_one")
        z["extra"] = z.did * z.n * z.days
        z["baseline"] = z.base * z.n * z.days
        den = z.baseline.sum()
        if den <= 0: raise ValueError("Positive baseline required for ROI")
        draws = rr.integers(0, len(z), size=(2000, len(z)))
        boot = z.extra.to_numpy()[draws].sum(axis=1) / z.baseline.to_numpy()[draws].sum(axis=1)
        # ci_lo/ci_hi (not lo/hi) so the CLV tier cut-off `hi` from Section 5 is never overwritten.
        ci_lo, ci_hi = np.quantile(boot, [.025, .975]); base_lift = z.extra.sum() / den
        lift_rows.append(dict(seg=sg, segment=NAMES[sg], worst=min(ci_lo, base_lift), base=base_lift, best=max(ci_hi, base_lift)))
    lifts = pd.DataFrame(lift_rows)
    # Gross margin: 22% central, rounded from Kroger FY2024. Use 20.5% and 23.5% for the conservative and optimistic cases, respectively (1.5 percentage points either side), based on the 10-year margin spread.
    # Distribution cost (printing and postage): $0.90 per contact based on USPS benchmarks. Vary by +/-10%: $0.99 for the conservative case and $0.81 for the optimistic case.
    # Preparation cost (fixed creative cost): $100 per campaign across cases, rounded from 4 hours of design at $24.21/hour ($96.84) based on USPS outsourced design estimates.
    assumptions = pd.DataFrame([
        dict(case="worst", margin=.205, distribution=.99, fixed_per_campaign=100.),
        dict(case="base", margin=.22, distribution=.90, fixed_per_campaign=100.),
        dict(case="best", margin=.235, distribution=.81, fixed_per_campaign=100.)])
    # SALES_VALUE is the recorded value after transaction discounts, so coupon face value
    # is not subtracted again. Unobserved manufacturer funding and processing costs remain
    # limitations. Assume an existing customer mailing list, so no list purchase is needed.
    parts = []; totals = []
    for a in assumptions.itertuples(index=False):
        z = so_roi.merge(o_roi[["c", "days"]], on="c", validate="many_to_one")
        z = z.merge(lifts[["seg", a.case]].rename(columns={a.case: "lift"}), on="seg", validate="many_to_one")
        z["segment"] = z.seg.map(NAMES)
        z["baseline_sales"] = z.base * z.n * z.days
        z["incremental_sales"] = z.baseline_sales * z.lift
        # Use Section 6's matched dollar lift in the central case.
        if a.case == "base": z["incremental_sales"] = z.did * z.n * z.days
        events = e.groupby(["c", "seg"]).redemptions.sum().rename("events")
        z = z.merge(events, left_on=["c", "seg"], right_index=True, validate="one_to_one")
        z["observed_redemptions"] = z.events
        z["distribution_cost"] = z.n * a.distribution
        z["incremental_contribution"] = z.incremental_sales * a.margin
        z["case"] = a.case; parts.append(z)
        for strategy, zz in [
            (BROAD_STRATEGY, z),
            (TARGETED_STRATEGY, z[z.segment.isin(RESPONSIVE)]),
        ]:
            sends = int(zz.n.sum()); campaigns = zz.c.nunique()
            fixed = campaigns * a.fixed_per_campaign
            costs = zz.distribution_cost.sum() + fixed
            sales = zz.incremental_sales.sum(); contribution = sales * a.margin
            profit = contribution - costs; baseline_sales = zz.baseline_sales.sum()
            totals.append(dict(case=a.case, strategy=strategy, recipient_campaign_pairs=sends,
                unique_households=e[e.segment.isin(zz.segment.unique())].household_key.nunique(),
                campaigns=campaigns, baseline_sales=baseline_sales, incremental_sales=sales,
                incremental_contribution=contribution, distribution_cost=zz.distribution_cost.sum(),
                fixed_cost=fixed, campaign_cost=costs,
                net_incremental_profit=profit, roi_pct=100 * profit / costs if costs else np.nan,
                breakeven_incremental_sales=costs / a.margin,
                breakeven_uplift_pct=100 * costs / (a.margin * baseline_sales),
                max_distribution_cost_per_send=(contribution - fixed) / sends,
                breakeven_margin=costs / sales if sales > 0 else np.nan))
    roi = pd.DataFrame(totals)
    # Each segment's incremental contribution after its own distribution cost. The preparation cost (fixed creative cost) is shared by all segments and identical under both strategies, so it is left out of this view.
    segment_economics = pd.concat(parts).groupby(["case", "segment"], as_index=False).agg(
        contacts=("n", "sum"), incremental_sales=("incremental_sales", "sum"),
        incremental_contribution=("incremental_contribution", "sum"),
        distribution_cost=("distribution_cost", "sum"))
    segment_economics["contribution_after_distribution"] = (
        segment_economics.incremental_contribution - segment_economics.distribution_cost)
    _broad = roi[roi.strategy == BROAD_STRATEGY].set_index("case")
    assert np.allclose(segment_economics.groupby("case").contribution_after_distribution.sum().reindex(_broad.index),
                       _broad.net_incremental_profit + _broad.fixed_cost)
    print("\nPlanning inputs:\n" + assumptions.to_string(index=False))
    print("\nSegment uplift assumptions (%):\n" + lifts.set_index("segment")[["worst", "base", "best"]].mul(100).round(2).to_string())
    print("\nScenario results:\n" + roi[["case", "strategy", "recipient_campaign_pairs", "incremental_sales", "campaign_cost", "net_incremental_profit", "roi_pct", "breakeven_uplift_pct"]].round(2).to_string(index=False))
    comparison = roi.pivot(index="case", columns="strategy", values=["net_incremental_profit", "roi_pct", "campaign_cost"])
    comparison.columns = [a + " | " + b for a, b in comparison.columns]
    comparison["targeting_profit_advantage"] = comparison[f"net_incremental_profit | {TARGETED_STRATEGY}"] - comparison[f"net_incremental_profit | {BROAD_STRATEGY}"]
    comparison["targeting_roi_advantage_pp"] = comparison[f"roi_pct | {TARGETED_STRATEGY}"] - comparison[f"roi_pct | {BROAD_STRATEGY}"]
    print("\nTargeting minus all: positive profit difference favours targeting.\n" + comparison[["targeting_profit_advantage", "targeting_roi_advantage_pp"]].round(2).to_string())
    assert np.allclose(roi.net_incremental_profit, roi.incremental_contribution - roi.campaign_cost)
    assert np.allclose(roi.roi_pct, 100 * roi.net_incremental_profit / roi.campaign_cost)

In [ ]:
scenario_labels = {"worst": "Conservative", "base": "Central", "best": "Optimistic"}

base_strategy_rows = roi.loc[roi.case == "base"].set_index("strategy")
strategy_definitions = pd.DataFrame({
    "Strategy": [BROAD_STRATEGY, TARGETED_STRATEGY],
    "Included households": [
        "All recorded recipients with a behavioural segment and usable matched comparison",
        "Big-basket regulars (prioritise) and Frequent top-up shoppers (controlled pilot) only",
    ],
    "Household–campaign contacts": [
        base_strategy_rows.loc[BROAD_STRATEGY, "recipient_campaign_pairs"],
        base_strategy_rows.loc[TARGETED_STRATEGY, "recipient_campaign_pairs"],
    ],
    "Campaigns retained": [
        base_strategy_rows.loc[BROAD_STRATEGY, "campaigns"],
        base_strategy_rows.loc[TARGETED_STRATEGY, "campaigns"],
    ],
})

input_roles = pd.DataFrame({
    "Input": [
        "Matched spending uplift", "Lower/upper uplift range", "Gross margin",
        "Distribution cost", "Fixed creative cost", "Unobserved costs",
    ],
    "Role": [
        "Empirical input from the matched campaign-lift analysis",
        "Empirical uncertainty from campaign-level resampling",
        "Scenario assumption; not an observed segment or category margin",
        "Scenario assumption per household–campaign contact",
        f"Scenario assumption applied to each of the {len(ROI_CAMPAIGNS)} Type B/C campaigns",
        "Coupon processing, some funding arrangements and placement costs are excluded",
    ],
})

assumptions_display = assumptions.assign(
    Scenario=lambda frame: frame.case.map(scenario_labels),
    **{"Uplift basis": ["Lower resampled estimate", "Central matched estimate", "Upper resampled estimate"]},
).rename(columns={
    "margin": "Assumed gross margin",
    "distribution": "Distribution cost per contact",
    "fixed_per_campaign": "Fixed cost per campaign",
})[["Scenario", "Uplift basis", "Assumed gross margin",
    "Distribution cost per contact", "Fixed cost per campaign"]]

uplift_display = lifts[["segment", "worst", "base", "best"]].rename(columns={
    "segment": "Customer segment", "worst": "Conservative uplift",
    "base": "Central uplift", "best": "Optimistic uplift",
})

broad_results = roi.loc[roi.strategy == BROAD_STRATEGY].set_index("case")
targeted_results = roi.loc[roi.strategy == TARGETED_STRATEGY].set_index("case")
financial_comparison = pd.DataFrame(index=["worst", "base", "best"])
financial_comparison["Scenario"] = financial_comparison.index.map(scenario_labels)
financial_comparison["Broad contacts"] = broad_results.recipient_campaign_pairs
financial_comparison["Selective contacts"] = targeted_results.recipient_campaign_pairs
financial_comparison["Contacts avoided"] = (
    financial_comparison["Broad contacts"] - financial_comparison["Selective contacts"]
)
financial_comparison["Contact reduction"] = (
    financial_comparison["Contacts avoided"] / financial_comparison["Broad contacts"]
)
financial_comparison["Broad incremental sales"] = broad_results.incremental_sales
financial_comparison["Selective incremental sales"] = targeted_results.incremental_sales
financial_comparison["Broad campaign cost"] = broad_results.campaign_cost
financial_comparison["Selective campaign cost"] = targeted_results.campaign_cost
financial_comparison["Campaign-cost saving"] = (
    financial_comparison["Broad campaign cost"] - financial_comparison["Selective campaign cost"]
)
financial_comparison["Broad estimated profit"] = broad_results.net_incremental_profit
financial_comparison["Selective estimated profit"] = targeted_results.net_incremental_profit
financial_comparison["Selective profit advantage"] = (
    financial_comparison["Selective estimated profit"] - financial_comparison["Broad estimated profit"]
)
financial_comparison["Broad ROI"] = broad_results.roi_pct / 100
financial_comparison["Selective ROI"] = targeted_results.roi_pct / 100
financial_comparison["Selective break-even uplift"] = targeted_results.breakeven_uplift_pct / 100
financial_comparison["Decision"] = np.where(
    financial_comparison["Selective profit advantage"] > 0,
    "Selective distribution financially preferred in this scenario",
    "Broad distribution financially preferred in this scenario",
)
financial_comparison = financial_comparison.reset_index(drop=True)

central_row = financial_comparison.loc[financial_comparison.Scenario == "Central"].iloc[0]
interpretation_table = pd.DataFrame({
    "Question": [
        "Why can selective incremental sales exceed broad incremental sales?",
        "What creates the direct cost saving?",
        "What does the result establish?",
    ],
    "Answer": [
        "The excluded Deal-seeker and Light-shopper contacts have negative central lift estimates under Type B/C, so removing them can increase the combined estimate rather than merely reduce mailing volume.",
        f"The same campaigns are retained, so fixed cost is unchanged. Avoiding {central_row['Contacts avoided']:,.0f} contacts reduces central-scenario distribution cost by ${central_row['Campaign-cost saving']:,.0f}.",
        "This is a retrospective planning scenario. It supports a controlled future targeting test; it is not observed campaign profit or proof of future ROI.",
    ],
})

display(strategy_definitions.style.hide(axis="index").format({
    "Household–campaign contacts": "{:,.0f}", "Campaigns retained": "{:,.0f}",
}).set_properties(
    subset=["Strategy", "Included households"],
    **{"text-align": "left", "white-space": "normal", "max-width": "520px"}
).set_caption("Strategies compared — the broad strategy covers matched recorded recipients, not every household"))
display(input_roles.style.hide(axis="index").set_properties(
    subset=["Input", "Role"],
    **{"text-align": "left", "white-space": "normal", "max-width": "560px"}
).set_caption("Separate empirical campaign-response inputs from financial assumptions"))
display(assumptions_display.style.hide(axis="index").format({
    "Assumed gross margin": "{:.1%}", "Distribution cost per contact": "${:.2f}",
    "Fixed cost per campaign": "${:,.0f}",
}).set_caption("Conservative, central and optimistic financial scenarios"))
display(uplift_display.style.hide(axis="index").format({
    "Conservative uplift": "{:+.1%}", "Central uplift": "{:+.1%}",
    "Optimistic uplift": "{:+.1%}",
}).set_caption("Segment uplift inputs derived from matched campaign estimates and resampling"))
display(financial_comparison.style.hide(axis="index").format({
    "Broad contacts": "{:,.0f}", "Selective contacts": "{:,.0f}", "Contacts avoided": "{:,.0f}",
    "Contact reduction": "{:.1%}", "Broad incremental sales": "${:,.0f}",
    "Selective incremental sales": "${:,.0f}", "Broad campaign cost": "${:,.0f}",
    "Selective campaign cost": "${:,.0f}", "Campaign-cost saving": "${:,.0f}",
    "Broad estimated profit": "${:,.0f}", "Selective estimated profit": "${:,.0f}",
    "Selective profit advantage": "${:+,.0f}", "Broad ROI": "{:.1%}",
    "Selective ROI": "{:.1%}", "Selective break-even uplift": "{:.2%}",
}).set_properties(
    subset=["Scenario", "Decision"],
    **{"text-align": "left", "white-space": "normal", "max-width": "380px"}
).set_caption("Financial comparison: broad matched-recipient versus responsive-segment distribution (Type B/C campaigns)"))
display(interpretation_table.style.hide(axis="index").set_properties(
    subset=["Question", "Answer"],
    **{"text-align": "left", "white-space": "normal", "max-width": "650px"}
).set_caption("How to interpret the financial comparison without overstating it"))

segment_order = [NAMES[s] for s in SEG_ORDER]
segment_economics_display = segment_economics.pivot(
    index="segment", columns="case", values="contribution_after_distribution"
).reindex(index=segment_order, columns=["worst", "base", "best"]).rename(columns=scenario_labels)
segment_economics_display.insert(0, "Contacts", segment_economics.loc[segment_economics.case == "base"]
                                 .set_index("segment").contacts.reindex(segment_order))
display(segment_economics_display.reset_index().rename(columns={"segment": "Segment"}).style.hide(axis="index").format({
    "Contacts": "{:,.0f}", "Conservative": "${:+,.0f}", "Central": "${:+,.0f}", "Optimistic": "${:+,.0f}",
}).set_caption(
    "Incremental contribution minus distribution cost by segment, Type B/C campaigns "
    "(the shared fixed creative cost is excluded)"
))


### 8.2 GE–McKinsey-style segment prioritisation matrix

This GE–McKinsey-style summary combines the two dimensions used in the final Type B/C decision:

- **Segment attractiveness:** contribution CLV per household, total segment contribution CLV and Top-to-Top retention. Contribution CLV uses the assumed 22% margin; the preceding table also reports revenue CLV without that margin assumption.
- **Campaign responsiveness:** matched total-spending lift for the 25 Type B/C campaigns and its uncertainty interval across campaigns. A range entirely below zero is labelled a negative estimate, not proof that coupons reduce spending.

Each point is a customer segment—not a segment–category combination. The matrix summarises the evidence already calculated; it does not create a new causal estimate.

In [ ]:
# GE–McKinsey-style segment prioritisation matrix.
priority_matrix = type_bc_targeting_decision.copy()
priority_matrix["Attractiveness score"] = (
    priority_matrix["Contribution CLV per household (assumed 22% margin)"].rank(pct=True)
    + priority_matrix["Segment contribution CLV (assumed 22% margin)"].rank(pct=True)
    + priority_matrix["Top-to-Top transition"].rank(pct=True)
) / 3
priority_matrix["Segment attractiveness"] = np.select(
    [priority_matrix["Attractiveness score"] >= 0.65,
     priority_matrix["Attractiveness score"] >= 0.35],
    ["High", "Medium"], default="Low",
)
priority_matrix["Campaign responsiveness"] = np.select(
    [priority_matrix["95% lower"] > 0, priority_matrix["95% upper"] < 0],
    ["Reliable positive", "Negative estimate"], default="Uncertain",
)

priority_matrix_display = priority_matrix[[
    "Segment", "Segment attractiveness", "Revenue CLV per household",
    "Contribution CLV per household (assumed 22% margin)",
    "Segment contribution CLV (assumed 22% margin)",
    "Top-to-Top transition", "Campaign responsiveness", "Matched spending lift",
    "95% lower", "95% upper", "Recommended Type B/C treatment",
]].rename(columns={"Recommended Type B/C treatment": "Suggested action"})

display(priority_matrix_display.style.hide(axis="index").format({
    "Revenue CLV per household": "${:,.0f}",
    "Contribution CLV per household (assumed 22% margin)": "${:,.0f}",
    "Segment contribution CLV (assumed 22% margin)": "${:,.0f}",
    "Top-to-Top transition": "{:.1%}", "Matched spending lift": "{:+.1%}",
    "95% lower": "{:+.1%}", "95% upper": "{:+.1%}",
}).set_properties(
    subset=["Segment", "Segment attractiveness", "Campaign responsiveness", "Suggested action"],
    **{"text-align": "left", "white-space": "normal", "max-width": "330px"}
).set_caption(
    "Segment prioritisation: customer attractiveness × campaign responsiveness (Type B/C lift)"
))

fig, ax = plt.subplots(figsize=(10, 6), dpi=140)
bubble_size = 250 + 750 * (
    priority_matrix["Segment contribution CLV (assumed 22% margin)"]
    / priority_matrix["Segment contribution CLV (assumed 22% margin)"].max()
)
colours = priority_matrix["Campaign responsiveness"].map({
    "Reliable positive": "#16A34A", "Uncertain": "#F59E0B", "Negative estimate": "#DC2626",
})
xerr = np.vstack([
    priority_matrix["Matched spending lift"] - priority_matrix["95% lower"],
    priority_matrix["95% upper"] - priority_matrix["Matched spending lift"],
])
ax.errorbar(
    priority_matrix["Matched spending lift"], priority_matrix["Attractiveness score"],
    xerr=xerr, fmt="none", ecolor="#6B7280", elinewidth=1.2, capsize=3, alpha=0.75,
)
ax.scatter(
    priority_matrix["Matched spending lift"], priority_matrix["Attractiveness score"],
    s=bubble_size, c=colours, alpha=0.82, edgecolor="white", linewidth=1.2,
)
for _, row in priority_matrix.iterrows():
    ax.annotate(
        row["Segment"],
        (row["Matched spending lift"], row["Attractiveness score"]),
        xytext=(6, 6), textcoords="offset points", fontsize=9, weight="bold",
    )
ax.axvline(0, color="#6B7280", linestyle="--", linewidth=1)
ax.axhline(0.65, color="#D1D5DB", linestyle=":", linewidth=1)
ax.axhline(0.35, color="#D1D5DB", linestyle=":", linewidth=1)
ax.set_xlabel("Matched campaign spending lift, Type B/C campaigns")
ax.set_ylabel("Segment attractiveness score")
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylim(0, 1.12)
ax.set_title("Type B/C priority matrix: value × campaign response", weight="bold")
ax.text(ax.get_xlim()[0], 0.91, "High attractiveness", color="#4B5563", fontsize=9)
ax.text(ax.get_xlim()[0], 0.50, "Medium attractiveness", color="#4B5563", fontsize=9)
ax.text(ax.get_xlim()[0], 0.12, "Low attractiveness", color="#4B5563", fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## Bottom line

The script tests whether course methods produce stable, out-of-time, or matched-comparison evidence before they are used in recommendations:

- segmentation is useful if the four-cluster solution is separated and stable;
- campaign diagnostics apply the documented delivery mechanics: Type A lists a personalised coupon pool, while Types B/C list the complete issued bundle;
- Type B/C category screening combines pre-campaign affinity with matched recipient-versus-non-recipient spending evidence; only combinations supported across periods are higher-confidence pilot candidates;
- redemption-based department and commodity bundle checks remain supporting utilisation evidence, not proof of incremental value;
- Markov CLV provides a coherent value projection, but inherits stationary-transition and margin assumptions;
- matched campaign lift is the primary input for selecting Type B/C audiences, while CLV determines the scale of investment in responsive segments;
- the decision table, ROI and priority matrix use the 25 Type B/C campaigns only, because Type A is kept unchanged;
- display and mailer analysis compares sales of coupon-eligible products under display-only, mailer-only and combined support (two-way fixed effects on a 5% sample); it is supplementary evidence, not a targeting rule;
- ROI translates those estimates into a targeting decision and exposes the assumptions required to break even.

When presenting results, distinguish **descriptive utilisation** findings, **observational campaign comparisons**, **causal assumptions**, and **financial scenario** assumptions. That distinction is the main analytical value of these checks.

## References

dunnhumby. (n.d.-a). *The Complete Journey* [Data set]. Kaggle. Retrieved November 12, 2026, from https://www.kaggle.com/datasets/frtgnn/dunnhumby-the-complete-journey

dunnhumby. (n.d.-b). *The Complete Journey user guide*. Retrieved November 12, 2026, from https://www.dunnhumby.com/source-files/

The Kroger Co. (2025). *2024 annual report (Form 10-K).* U.S. Securities and Exchange Commission. https://www.sec.gov/Archives/edgar/data/56873/000155837025004267/kr-20250201x10k.htm

United States Postal Service. (n.d.). *Estimate the cost of your next direct mail campaign.* USPS Delivers. Retrieved November 12, 2026, from https://www.uspsdelivers.com/direct-mail-cost-calculator/